<a href="https://colab.research.google.com/github/xyosho/Proyecto-Homologacion-ML/blob/main/sprints_1_a_5_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprints 1 a 5 en un solo cuaderno
**Proyecto:** Modelo Predictivo de Segmentación de Mercado y Propensión a la Matrícula · Unicervantes

Este cuaderno reúne, en orden, los cuadernos de los Sprints 1 a 5 para ejecutarlos de una sola vez:
- La **preparación** (librerías, módulos y conexión segura) se hace **una sola vez**, al inicio.
- Los **datos** se cargan una vez al comenzar el Sprint 2 y los **modelos finales**, una vez al comenzar el Sprint 4.
- Cada sprint empieza con la marca **COMIENZA EL SPRINT N** y termina con **FIN DEL SPRINT N**.

| Sprint | Tema | Entrega |
|---|---|---|
| 1 | Definición del problema, datos y análisis exploratorio | jueves 22 de octubre de 2026 |
| 2 | Ingeniería de características y regresión logística | jueves 29 de octubre de 2026 |
| 3 | Modelado: regresión logística mejorada con el NLP, frente a Random Forest y XGBoost | jueves 5 de noviembre de 2026 |
| 4 | Machine Learning explicable con SHAP y LIME | jueves 12 de noviembre de 2026 |
| 5 | Validación con datos no vistos e impacto comercial | jueves 19 de noviembre de 2026 |

# Preparación (una sola vez)
Estas celdas dejan listo el entorno para los Sprints 1 a 5: rutas del proyecto, librerías y la conexión segura con
Supabase (las credenciales se leen del archivo `.env` o de los Secretos de Colab; nunca se escriben aquí).

## Antes de empezar (Google Colab)
Las credenciales **no** se escriben en ninguna celda; se guardan en los **Secretos** de Colab:

1. Abra el panel **Secretos** (icono de llave, a la izquierda).
2. Cree `SUPABASE_URL` (la URL del proyecto, `https://….supabase.co`) y `SUPABASE_KEY` (la clave **publicable**,
   `sb_publishable_…`; nunca la secreta).
3. Active **Acceso del notebook** en los dos secretos.
4. Ejecute todo: *Entorno de ejecución → Ejecutar todas*. Si aparece un aviso para dar acceso a los secretos, acéptelo.

**¿Sin clave de Supabase?** No hay que subir ningún archivo: el cuaderno lee solo la copia de los datos que está en
GitHub (`data/processed` de la rama `Sprint-5` del repositorio) y dice de dónde salieron los datos.

Las celdas que empiezan con `%%writefile` crean en Colab una copia exacta del código del repositorio (carpeta `src/`).

In [ ]:
!pip -q install scikit-learn==1.9.0 supabase==2.33.0 xgboost==3.4.1 shap==0.54.1 lime==0.2.0.1

In [ ]:
%%writefile comun.py
"""Versión para Google Colab de src/comun.py: la configuración del proyecto (config/reglas_negocio.yaml)
viene incrustada porque Colab no tiene el repositorio. Generado por src/s06_construir_cuadernos.py."""
from pathlib import Path

RAIZ = Path.cwd()
DATA_PROCESSED = RAIZ / "data" / "processed"

_CONFIG = {'campus': {'nombre': 'Campus Unicervantes',
            'direccion': 'Calle 209 No. 104-15, vía Arrayanes, Bogotá',
            'lat': 4.790438,
            'lon': -74.066288,
            'fuente': 'Dirección: Acuerdo 002/2025 (membrete). Coordenada exacta de la placa Calle 209 # '
                      '104-15 (CL 209 104 15, lote 009140001001) en Placa Domiciliaria de Catastro Bogotá '
                      '(UAECD), consultada 23-oct-2026. Antes se usaba una coordenada aproximada de '
                      'OpenStreetMap (4.7895, -74.0660), a unos 100 m.'},
 'tarifas': {'AD': {2024: {'valor': 4419000,
                           'fuente': 'Precio de matrícula según recibos de pago: orden de pago Pregrado '
                                     "2024-2, concepto 'MAT ING ANALITICA DE DATOS NUEVOS'."},
                    2025: {'valor': 4658000,
                           'fuente': 'Precio de matrícula según recibos de pago 2025-1 y 2025-2: $3.260.600 '
                                     'con beca del 30% => 3.260.600 / 0,70; coincide con la versión 2025 de '
                                     'la página del programa.'},
                    2026: {'valor': 4914000,
                           'fuente': 'Acuerdo 002 del 11 de noviembre de 2025 del Consejo Directivo, art. 2 '
                                     '(valores de matrícula 2026).'},
                    2027: {'valor': 4914000,
                           'fuente': 'supuesto: S-01 (se mantiene el valor vigente hasta que se publique el '
                                     'acuerdo 2027).'}},
             'SSIA': {2026: {'valor': 4950000,
                             'fuente': 'Oferta académica oficial '
                                       'unicervantes.edu.co/admision/oferta-academica (consultada '
                                       '19-oct-2026).'},
                      2027: {'valor': 4950000, 'fuente': 'supuesto: S-01.'}}},
 'becas': {'SABER_11': {'nombre': 'Beca Saber 11',
                        'umbral_icfes': 310,
                        'pct_primer_periodo': 0.35,
                        'pct_siguientes': 0.3,
                        'promedio_minimo': 4.0,
                        'fuente': 'Admisiones (corrección del 31-oct-2026): 35% el 1er semestre y 30% desde '
                                  'el 2º; coincide con los recibos (2024-2: 35%; 2025 y 2026: 30%). Página '
                                  "oficial: 'Puntaje igual o superior a 310 ... 30% para toda la carrera por "
                                  "promedio académico'."},
           'MI_VOCACION': {'nombre': 'Beca Cervantes #MiVocación',
                           'pct_primer_periodo': 0.3,
                           'pct_siguientes': 0.2,
                           'promedio_minimo': 4.0,
                           'fuente': "Admisiones ('30% para el primer periodo académico y en adelante según "
                                     "la tabla relacionada'; se mantiene con promedio acumulado >= 4.0 y se "
                                     "pierde por sanciones); landing del programa ('30% 1er semestre y 20% "
                                     "el resto de la carrera')."},
           'NINGUNA': {'nombre': 'Sin beca',
                       'pct_primer_periodo': 0.0,
                       'pct_siguientes': 0.0,
                       'promedio_minimo': None},
           'acumulables': False},
 'particion': {'entrenamiento': ['2024-1', '2024-2', '2025-1', '2025-2'],
               'validacion': ['2026-1'],
               'prueba': ['2026-2'],
               'campana': ['2027-1']},
 'caracteristicas': {'bandas_icfes': [{'desde': 0, 'hasta': 250, 'segmento': 'BAJO'},
                                      {'desde': 250, 'hasta': 310, 'segmento': 'MEDIO'},
                                      {'desde': 310, 'hasta': 350, 'segmento': 'ALTO'},
                                      {'desde': 350, 'hasta': 501, 'segmento': 'SUPERIOR'}],
                     'zonas_distancia_km': {'CERCA': 10, 'MEDIA': 20},
                     'canal_agrupado': {'EVENTO': ['EXPOESTUDIANTES', 'CAMPUS_DAY'],
                                        'RELACIONAL': ['REFERIDO', 'INMERSION', 'COLEGIO_CONVENIO'],
                                        'DIGITAL': ['SITIO_WEB', 'REDES_SOCIALES', 'LLAMADA_WHATSAPP']},
                     'estrato_grupo': {1: 'BAJO', 2: 'BAJO', 3: 'MEDIO', 4: 'ALTO', 5: 'ALTO', 6: 'ALTO'},
                     'kmeans': {'k_min': 3,
                                'k_max': 7,
                                'tolerancia_silueta': 0.01,
                                'semilla': 20261029,
                                'variables': ['estrato',
                                              'log_distancia',
                                              'punt_global',
                                              'edad_al_inscribirse',
                                              'colegio_privado']},
                     'nlp': {'ngramas': [1, 2],
                             'min_df': 3,
                             'alphas': [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0],
                             'pliegues': 5,
                             'semilla': 20261112}},
 'modelado': {'semilla': 20261105,
              'umbral_clasificacion': 0.5,
              'segmentos_propension': {'ALTA': 0.6, 'BAJA': 0.35},
              'sprint2_logistica': {'C': 1.0},
              'grillas': {'logistica': {'C': [0.01, 0.1, 1.0, 10.0],
                                        'l1_ratio': [0.0, 1.0],
                                        'solver': ['saga']},
                          'random_forest': {'max_depth': [4, 6, 8, None],
                                            'min_samples_leaf': [5, 20],
                                            'max_features': ['sqrt', 0.5]},
                          'xgboost': {'n_estimators': [200, 400],
                                      'max_depth': [2, 3, 4],
                                      'learning_rate': [0.03, 0.1],
                                      'min_child_weight': [5]}}},
 'impacto': {'seguimiento': {'capacidad_pct': 0.3, 'efecto_maximo': 0.1}},
 'escala_desempeno_bachillerato': {'SUPERIOR': 4.6, 'ALTO': 4.0, 'BASICO': 3.0},
 'fondos_educacion_municipal': {'FOES_CHIA': {'ubicacion_id': 25175,
                                              'nombre': 'FOES - Fondo para el Fomento de la Educación '
                                                        'Superior de Chía',
                                              'norma': 'Acuerdo 172 de 2020; Resolución 2114 de 2021',
                                              'convenio_unicervantes': 'VERIFICADO',
                                              'apoyo_semestral_cop': 1000000,
                                              'requisitos': 'Residente de Chía (certificado de la JAC), '
                                                            'registro en el SISBÉN de Chía, certificado de '
                                                            'notas, Saber 11 y recibo de matrícula',
                                              'renovacion': 'Cada semestre (abril a octubre) según el '
                                                            'promedio del semestre anterior fijado en el '
                                                            'convenio',
                                              'fuente': 'Alcaldía de Chía (2022). Conozca el programa FOES: '
                                                        'Unicervantes en convenio (n.º 15); apoyo educativo '
                                                        'desde $1.000.000 en carreras presenciales.',
                                              'url': 'https://www.chia-cundinamarca.gov.co/2022/educacion/PROGRAMA%20FOES%20ACTUALIZADO%2022-09-2022.pdf'},
                                'FOES_SOPO': {'ubicacion_id': 25758,
                                              'nombre': 'FOES - Fondo Especial para el Fomento de la '
                                                        'Educación de Sopó',
                                              'norma': 'No publicada en la página consultada',
                                              'convenio_unicervantes': 'NO_VERIFICADO',
                                              'apoyo_semestral_cop': 1000000,
                                              'requisitos': 'Residente de Sopó, admitido o matriculado en un '
                                                            'programa de educación superior',
                                              'renovacion': 'No publicada',
                                              'fuente': 'Alcaldía de Sopó (2022, actualizado 2026). FOES: '
                                                        'apoyo de matrícula, transporte y sostenimiento. '
                                                        'Monto: supuesto S-31.',
                                              'url': 'https://www.sopo-cundinamarca.gov.co/educacion/publicaciones/290/foes-fondo-especial-para-el-fomento-de-la-educacion/'},
                                'FOEST_TOCANCIPA': {'ubicacion_id': 25817,
                                                    'nombre': 'FOEST - Fondo Especial de Educación Superior '
                                                              'de Tocancipá',
                                                    'norma': 'Aprobado por el Concejo Municipal (acuerdo no '
                                                             'consultado)',
                                                    'convenio_unicervantes': 'NO_VERIFICADO',
                                                    'apoyo_semestral_cop': 1000000,
                                                    'requisitos': 'Residente de Tocancipá; becas para '
                                                                  'mejores puntajes Saber 11, artistas y '
                                                                  'deportistas; descuentos por convenio',
                                                    'renovacion': 'No publicada',
                                                    'fuente': 'Alcaldía de Tocancipá (2017). $1.750 millones '
                                                              'para la educación superior. Monto por '
                                                              'estudiante: supuesto S-31.',
                                                    'url': 'https://www.tocancipa-cundinamarca.gov.co/sala-prensa/noticias-1/1-750-millones-para-educacion-superior-tocancipa'}},
 'repositorio': {'github': 'xyosho/Proyecto-Homologacion-ML', 'rama_datos': 'Sprint-5'},
 'generador': {'matricula': {'muestra_ejemplo': {'aspirantes': 76,
                                                 'matriculados': 43,
                                                 'por_periodo': {'2024-1': [22, 16],
                                                                 '2024-2': [9, 4],
                                                                 '2025-1': [8, 6],
                                                                 '2025-2': [9, 3],
                                                                 '2026-1': [28, 14]}},
                             'beta_descuento': 3.0,
                             'beta_estrato': {1: -0.5, 2: -0.15, 3: 0.0, 4: 0.1, 5: 0.0, 6: -0.3},
                             'beta_log_distancia': -0.35,
                             'beta_icfes_cuadratico': -0.25,
                             'beta_canal': {'EXPOESTUDIANTES': -0.3,
                                            'SITIO_WEB': 0.0,
                                            'REDES_SOCIALES': -0.1,
                                            'REFERIDO': 0.5,
                                            'CAMPUS_DAY': 0.3,
                                            'LLAMADA_WHATSAPP': -0.2,
                                            'INMERSION': 0.8,
                                            'COLEGIO_CONVENIO': 0.3},
                             'beta_entrevista': 0.6,
                             'beta_programa_ssia': 0.1,
                             'beta_periodo_2': -0.1,
                             'beta_foes': 0.5},
               'entrevista_texto': {'secciones': {'INTERES_PROGRAMA': 'Interés por el programa académico',
                                                  'HISTORIA_VIDA': 'Historia de vida',
                                                  'INTERES_UNIVERSIDAD': 'Interés por la universidad'},
                                    'preguntas': [{'id': 1,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': '¿Por qué quiere estudiar esta carrera '
                                                            'profesional?'},
                                                  {'id': 2,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': 'Por favor defina su programa elegido'},
                                                  {'id': 3,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': '¿Qué es lo que más le gusta de la carrera?'},
                                                  {'id': 4,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': '¿Cómo se ve en un futuro como profesional?'},
                                                  {'id': 5,
                                                   'seccion': 'INTERES_PROGRAMA',
                                                   'texto': 'Si tuviera que elegir otra carrera, ¿cuál '
                                                            'sería? ¿Por qué?'},
                                                  {'id': 6,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': 'Cuéntenos sobre usted'},
                                                  {'id': 7,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': 'Ocupación de las personas con quien vive'},
                                                  {'id': 8,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Trabaja? ¿Cómo ha planeado financiar sus '
                                                            'estudios?'},
                                                  {'id': 9,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Cuáles considera que son sus cualidades y/o '
                                                            'habilidades? Mencionar mínimo 3'},
                                                  {'id': 10,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Qué aspectos considera que debe mejorar en '
                                                            'usted mismo? Mencionar mínimo 3'},
                                                  {'id': 11,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Cómo fue su rendimiento académico en el '
                                                            'colegio? (Asignaturas de mayor y menor '
                                                            'interés)'},
                                                  {'id': 12,
                                                   'seccion': 'HISTORIA_VIDA',
                                                   'texto': '¿Qué le gusta hacer en sus tiempos libres?'},
                                                  {'id': 13,
                                                   'seccion': 'INTERES_UNIVERSIDAD',
                                                   'texto': '¿Por qué quiere estudiar en Unicervantes?'}],
                                    'peso_intencion': 0.4,
                                    'de_vocacion': 1.0,
                                    'de_respuesta': 0.8,
                                    'corte_nivel': 0.6,
                                    'dias_hasta_entrevista': {'min': 1, 'max': 10}}}}


def cargar_config() -> dict:
    return _CONFIG


In [ ]:
%%writefile conexion_supabase.py
"""Conexión SEGURA de Python con Supabase para leer la tabla plana dataset_matricula.

Las credenciales NUNCA se escriben en el código ni se imprimen:
  - En el computador: archivo .env en la raíz del proyecto, excluido por .gitignore (plantilla: .env.example).
    Se puede crear con:  python src/s00_configurar_credenciales.py
  - En Google Colab: panel "Secretos" (icono de llave) con SUPABASE_URL y SUPABASE_KEY, con acceso al cuaderno.

SUPABASE_KEY es la clave PUBLICABLE del proyecto. En la base, la seguridad a nivel de fila (RLS) solo le permite
LEER dataset_matricula (sin datos personales); las otras 25 tablas están cerradas y nadie puede escribir con ella.
Así, aunque la clave se filtrara, solo se podría leer el dataset anonimizado.
"""
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

TABLA = "dataset_matricula"
VARIABLES = ("SUPABASE_URL", "SUPABASE_KEY")
TAMANO_PAGINA = 1000          # la API de Supabase entrega como máximo 1.000 filas por petición
TIEMPO_LIMITE_S = 30          # si la red no responde, se desiste en vez de quedarse esperando


def en_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def obtener_credenciales() -> tuple[str, str]:
    """Devuelve (url, clave) desde los Secretos de Colab o desde .env / variables de entorno.

    En Colab, la primera vez que un cuaderno lee un secreto aparece un aviso para darle acceso: hay que aceptarlo
    (mientras no se acepte, la celda espera). Los errores explican qué falta, nunca muestran los valores.
    """
    problemas = []
    if en_colab():
        from google.colab import userdata
        valores = {}
        for v in VARIABLES:
            try:
                valores[v] = (userdata.get(v) or "").strip()
            except Exception as e:                  # SecretNotFoundError / NotebookAccessError
                valores[v] = ""
                motivo = ("el cuaderno no tiene acceso: active 'Acceso del notebook'" if "Access" in type(e).__name__
                          else "no existe un secreto con ese nombre exacto")
                problemas.append(f"{v}: {motivo}")
        donde = "los Secretos de Colab (icono de llave, con 'Acceso del notebook' activado)"
    else:
        from dotenv import load_dotenv
        load_dotenv(Path(__file__).resolve().parents[1] / ".env")
        valores = {v: os.environ.get(v, "").strip() for v in VARIABLES}
        donde = "el archivo .env (python src/s00_configurar_credenciales.py o copia de .env.example)"
    problemas += [f"{v}: está vacío" for v, x in valores.items() if not x and not any(p.startswith(v) for p in problemas)]
    url, clave = valores["SUPABASE_URL"], valores["SUPABASE_KEY"]
    if url and not (url.startswith("https://") and url.rstrip("/").endswith(".supabase.co")):
        problemas.append("SUPABASE_URL: no es la URL del proyecto (debe ser https://<proyecto>.supabase.co; "
                         "no la cadena postgresql:// del pooler)")
    if clave.lower().startswith(("postgres://", "postgresql://")):
        problemas.append("SUPABASE_KEY: es la cadena del pooler; aquí va la clave PUBLICABLE (sb_publishable_...)")
    elif clave.startswith("sb_secret_"):
        problemas.append("SUPABASE_KEY: es la clave SECRETA; use la PUBLICABLE (sb_publishable_...)")
    if problemas:
        raise RuntimeError(f"Credenciales incompletas en {donde}: " + "; ".join(problemas) + ".")
    return url, clave


def crear_cliente(url: str | None = None, clave: str | None = None):
    """Cliente de la API de Supabase con tiempo límite. La clave se usa en memoria y no se muestra."""
    from supabase import ClientOptions, create_client
    if url is None or clave is None:
        url, clave = obtener_credenciales()
    return create_client(url, clave, options=ClientOptions(postgrest_client_timeout=TIEMPO_LIMITE_S))


def leer_dataset_matricula(cliente, tabla: str = TABLA) -> pd.DataFrame:
    """Lee la tabla completa en páginas de 1.000 filas (límite de la API), ordenada por inscripcion_id."""
    filas, inicio = [], 0
    while True:
        r = (cliente.table(tabla).select("*").order("inscripcion_id")
             .range(inicio, inicio + TAMANO_PAGINA - 1).execute())
        filas += r.data
        if len(r.data) < TAMANO_PAGINA:
            break
        inicio += TAMANO_PAGINA
    return pd.DataFrame(filas)


def probar_solo_lectura(cliente, tabla: str = TABLA) -> bool:
    """Intenta modificar la tabla con la clave publicable. Devuelve True si la base lo RECHAZA por permisos
    (código 42501 de PostgreSQL: permiso denegado). La fila -1 no existe, así que nada cambiaría de todos modos."""
    try:
        cliente.table(tabla).update({"punt_global": 0}).eq("inscripcion_id", -1).execute()
    except Exception as e:                           # APIError de la API de Supabase
        return getattr(e, "code", None) == "42501"
    return False


In [ ]:
%%writefile datos.py
"""Carga de las tablas planas para los sprints 2 a 6.

Fuente preferida: Supabase con la API de solo lectura (credenciales en .env o en los Secretos de Colab, ver
conexion_supabase.py). Si no hay credenciales, se usan los CSV que exporta el pipeline (data/processed), que son
idénticos (las pruebas lo comprueban): en el computador, los del repositorio; en Colab, donde no está el repositorio,
la copia que está en GitHub (data/processed de la rama del sprint), así que no hay que subir ningún archivo.
La función siempre dice de dónde salieron los datos.
"""
from __future__ import annotations

import io
import os
import urllib.request
from pathlib import Path

import pandas as pd

TABLAS = ("dataset_matricula", "dataset_entrevista", "resultado_campana_2027_1")   # el último solo en el Sprint 5
ENTEROS_CON_VACIOS = ("matriculado", "puntaje_vocacional", "matriculado_real")


def preparar_tipos(df: pd.DataFrame) -> pd.DataFrame:
    """La API devuelve JSON: se fijan los tipos para que local y Colab den exactamente lo mismo."""
    if df.empty:                                   # tabla sin filas: no hay tipos que fijar
        return df
    df = df.copy()
    for c in ENTEROS_CON_VACIOS:
        if c in df:
            df[c] = pd.to_numeric(df[c]).astype("Int64")
    for c in ("distancia_campus_km", "distancia_vial_km", "colegio_promedio_saber11", "pct_descuento_inicial"):
        if c in df:
            df[c] = pd.to_numeric(df[c]).astype(float)
    if "asesor_id" in df:
        df["asesor_id"] = df["asesor_id"].astype(int)
    return df.sort_values("inscripcion_id").reset_index(drop=True)


def _csv_local(tabla: str) -> Path:
    from comun import DATA_PROCESSED
    if tabla == "resultado_campana_2027_1":       # "sobre sellado" desde la Semana 1 (data/processed/oculto)
        return DATA_PROCESSED / "oculto" / "resultado_2027_1_sellado.parquet"
    return DATA_PROCESSED / f"{tabla}.csv"


def url_github(tabla: str) -> tuple[str, str]:
    """Dirección de la copia de la tabla en GitHub y la rama de donde sale (config/reglas_negocio.yaml -> repositorio;
    en Colab, cada cuaderno fija la rama de su sprint en la variable de entorno RAMA_DATOS_GITHUB)."""
    from comun import cargar_config
    repo = cargar_config()["repositorio"]
    rama = os.environ.get("RAMA_DATOS_GITHUB") or repo["rama_datos"]
    ruta = "data/processed/" + ("oculto/resultado_2027_1_sellado.parquet" if tabla == "resultado_campana_2027_1"
                                else f"{tabla}.csv")
    return f"https://raw.githubusercontent.com/{repo['github']}/{rama}/{ruta}", rama


def _leer_github(tabla: str) -> tuple[pd.DataFrame, str]:
    url, rama = url_github(tabla)
    with urllib.request.urlopen(url, timeout=120) as respuesta:
        contenido = io.BytesIO(respuesta.read())
    return (pd.read_parquet(contenido) if url.endswith(".parquet") else pd.read_csv(contenido)), rama


def cargar(tabla: str, fuente: str = "auto") -> tuple[pd.DataFrame, str]:
    """Devuelve (datos, origen). fuente: 'supabase', 'local', 'github' o 'auto' (Supabase si hay credenciales; si no,
    el CSV del repositorio si está en el computador, o su copia en GitHub, como en Colab)."""
    if tabla not in TABLAS:
        raise ValueError(f"Tabla desconocida: {tabla}")
    if fuente not in ("auto", "supabase", "local", "github"):
        raise ValueError(f"Fuente desconocida: {fuente}")
    aviso = ""
    if fuente in ("auto", "supabase"):
        try:
            from conexion_supabase import crear_cliente, leer_dataset_matricula
            df = leer_dataset_matricula(crear_cliente(), tabla=tabla)
            if df.empty:                       # la tabla existe pero no tiene filas (o la API no deja leerlas)
                raise RuntimeError(f"La tabla {tabla} está vacía en Supabase: cárguela con python src/s05_cargar_supabase.py.")
            return preparar_tipos(df), "Supabase (API de solo lectura)"
        except Exception as error:
            if fuente == "supabase":
                raise
            aviso = type(error).__name__
    sin_conexion = f" (sin conexión a Supabase: {aviso})" if fuente == "auto" else ""
    ruta = _csv_local(tabla)
    if fuente == "github" or (fuente == "auto" and not ruta.exists()):
        df, rama = _leer_github(tabla)
        return preparar_tipos(df), f"GitHub: copia de data/processed en la rama {rama} del repositorio{sin_conexion}"
    df = pd.read_parquet(ruta) if ruta.suffix == ".parquet" else pd.read_csv(ruta)
    return preparar_tipos(df), "CSV local exportado por el pipeline" + sin_conexion


def cargar_inscritos(fuente: str = "auto") -> tuple[pd.DataFrame, str]:
    """dataset_matricula con las respuestas de la entrevista (dataset_entrevista, texto por sección) unidas por inscripcion_id."""
    inscritos, origen = cargar("dataset_matricula", fuente)
    entrevista, _ = cargar("dataset_entrevista", fuente)
    if len(entrevista) != len(inscritos):     # p. ej. la tabla existe en Supabase pero aún no se cargó (src/s05)
        if fuente == "supabase":
            raise RuntimeError(f"dataset_entrevista tiene {len(entrevista)} filas y dataset_matricula {len(inscritos)}: "
                               "falta cargar la entrevista en Supabase (python src/s05_cargar_supabase.py).")
        entrevista, _ = cargar("dataset_entrevista", "local" if _csv_local("dataset_entrevista").exists() else "github")
    return inscritos.merge(entrevista, on="inscripcion_id", how="left", validate="one_to_one"), origen


In [ ]:
%%writefile nlp_vocacion.py
"""SPRINT 2 — Modelo NLP: el puntaje vocacional que se lee en las palabras del aspirante.

Entrada: dataset_entrevista, las respuestas abiertas de la entrevista de admisión unidas por sección
(Interés por el programa académico, Historia de vida e Interés por la universidad).
Objetivo: el puntaje vocacional que dio el entrevistador (1 a 5).
Modelo: TF-IDF por sección (palabras y pares de palabras, sin tildes ni palabras vacías) + regresión Ridge.

El resultado, puntaje_vocacional_nlp (1 a 5), es la vocación según las palabras del propio aspirante. No es una copia
del entrevistador: las palabras también reflejan lo que él no alcanza a ver (supuesto S-49), y eso es lo que la
regresión logística del Sprint 3 puede aprovechar.

Regla contra la FUGA DE DATOS: se ajusta SOLO con el ENTRENAMIENTO. Las filas de entrenamiento reciben predicciones
fuera de pliegue (validación cruzada de 5 pliegues): ninguna se puntúa con un modelo que ya vio su propia respuesta.

Funciones: ajustar(entrenamiento) -> dict · puntuar(nlp, tabla) -> Series · palabras_clave(nlp) · metricas(y, p)
"""
from __future__ import annotations

import re
import unicodedata
from functools import partial

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline

SECCIONES = {"texto_interes_programa": "Interés por el programa académico",
             "texto_historia_vida": "Historia de vida",
             "texto_interes_universidad": "Interés por la universidad"}

# Palabras vacías del español, sin tildes (el TF-IDF quita las tildes antes de compararlas). Se CONSERVAN las que
# cambian el sentido de una respuesta: "no", "nunca", "todavia", "ya", "muy", "sin" y las muletillas de duda ("pues").
STOPWORDS_ES = sorted({
    "a", "al", "algo", "algun", "alguna", "algunas", "alguno", "algunos", "ante", "antes", "asi", "aun", "cada",
    "como", "con", "contra", "cual", "cuales", "cuando", "de", "del", "desde", "donde", "durante", "e", "el", "ella",
    "ellas", "ellos", "en", "entre", "era", "eran", "es", "esa", "esas", "ese", "eso", "esos", "esta", "estaba",
    "estas", "este", "esto", "estos", "fue", "fueron", "ha", "han", "hasta", "hay", "la", "las", "le", "les", "lo",
    "los", "me", "mi", "mis", "nos", "nosotros", "o", "os", "otra", "otras", "otro", "otros", "para", "pero", "por",
    "que", "quien", "quienes", "se", "ser", "si", "sobre", "son", "su", "sus", "te", "ti", "tu", "tus", "u",
    "un", "una", "unas", "uno", "unos", "y", "yo"})


def configuracion() -> dict:
    try:
        from comun import cargar_config
        return cargar_config()
    except ImportError:
        return CONFIG_PROYECTO  # noqa: F821  (lo define una celda del cuaderno de Colab)


def analizar(texto: str, ngramas: tuple = (1, 2)) -> list[str]:
    """Cómo ve el modelo una respuesta: minúsculas y sin tildes, se parte en frases (cada respuesta termina en punto),
    se quitan las palabras vacías y se forman palabras y pares de palabras DENTRO de cada frase. Así un par nunca
    mezcla el final de una respuesta con el inicio de la siguiente."""
    t = "".join(c for c in unicodedata.normalize("NFKD", str(texto).lower()) if not unicodedata.combining(c))
    salida = []
    for frase in re.split(r"[.;:!?¡¿]+", t):
        palabras = [w for w in re.findall(r"\b\w\w+\b", frase) if w not in STOPWORDS_ES]
        for n in range(ngramas[0], ngramas[1] + 1):
            salida += [" ".join(palabras[i:i + n]) for i in range(len(palabras) - n + 1)]
    return salida


def _pipeline(cfg: dict) -> Pipeline:
    n = cfg["caracteristicas"]["nlp"]
    analizador = partial(analizar, ngramas=tuple(n["ngramas"]))
    tfidf = ColumnTransformer([(col, TfidfVectorizer(analyzer=analizador, min_df=n["min_df"], sublinear_tf=True), col)
                               for col in SECCIONES])
    return Pipeline([("tfidf", tfidf), ("ridge", RidgeCV(alphas=n["alphas"]))])


def con_texto(d: pd.DataFrame) -> pd.Series:
    """True si el inscrito presentó la entrevista (tiene respuestas)."""
    if not set(SECCIONES) <= set(d.columns):
        return pd.Series(False, index=d.index)
    return d[list(SECCIONES)].notna().all(axis=1)


def ajustar(entrenamiento: pd.DataFrame, cfg: dict | None = None) -> dict:
    """Aprende, solo con las entrevistas del ENTRENAMIENTO, a leer el puntaje vocacional en las palabras."""
    cfg = cfg or configuracion()
    n = cfg["caracteristicas"]["nlp"]
    e = entrenamiento[con_texto(entrenamiento)]
    X, y = e[list(SECCIONES)], e["puntaje_vocacional"].astype(float).to_numpy()
    modelo = _pipeline(cfg).fit(X, y)
    pliegues = KFold(n["pliegues"], shuffle=True, random_state=n["semilla"])
    fuera = np.clip(cross_val_predict(_pipeline(cfg), X, y, cv=pliegues), 1, 5)
    return {"modelo": modelo, "fuera_de_pliegue": pd.Series(fuera, index=e["inscripcion_id"].to_numpy()),
            "alpha": float(modelo.named_steps["ridge"].alpha_), "n_entrevistas": int(len(e)),
            "n_terminos": int(len(modelo.named_steps["tfidf"].get_feature_names_out())), "media_entrenamiento": float(y.mean())}


def puntuar(nlp: dict, d: pd.DataFrame) -> pd.Series:
    """puntaje_vocacional_nlp (1 a 5) de cada fila; vacío si no hubo entrevista. Las filas del entrenamiento usan su
    predicción fuera de pliegue; las demás (validación, prueba, campaña y aspirantes nuevos), el modelo completo."""
    s = pd.Series(np.nan, index=d.index, dtype=float)
    m = con_texto(d)
    if m.any():
        s[m] = np.clip(nlp["modelo"].predict(d.loc[m, list(SECCIONES)]), 1, 5)
        fuera = d["inscripcion_id"].map(nlp["fuera_de_pliegue"])
        s = fuera.where(fuera.notna() & m, s)
    return s.round(3)


def terminos(texto: str, cfg: dict | None = None) -> list[str]:
    """Los términos que el modelo extrae de un texto (para mostrar el preprocesamiento)."""
    return analizar(texto, tuple((cfg or configuracion())["caracteristicas"]["nlp"]["ngramas"]))


def palabras_clave(nlp: dict, k: int = 8) -> pd.DataFrame:
    """Los términos que más suben y más bajan el puntaje en cada sección: las palabras que determinan la vocación."""
    nombres = nlp["modelo"].named_steps["tfidf"].get_feature_names_out()
    pesos = pd.DataFrame({"columna": [x.split("__", 1)[0] for x in nombres], "termino": [x.split("__", 1)[1] for x in nombres],
                          "peso": nlp["modelo"].named_steps["ridge"].coef_})
    filas = []
    for col, nombre in SECCIONES.items():
        p = pesos[pesos["columna"] == col]
        for sentido, parte in (("sube", p.nlargest(k, "peso")), ("baja", p.nsmallest(k, "peso"))):
            filas += [{"seccion": nombre, "sentido": sentido, "termino": t, "peso": round(float(w), 4)}
                      for t, w in zip(parte["termino"], parte["peso"])]
    return pd.DataFrame(filas)


def metricas(y, p, media_entrenamiento: float | None = None) -> dict:
    """Qué tan bien lee el modelo el puntaje del entrevistador (y frente a predecir siempre el promedio)."""
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    base = float(np.mean(y) if media_entrenamiento is None else media_entrenamiento)
    return {"n": int(len(y)), "mae": float(np.mean(np.abs(y - p))), "mae_promedio": float(np.mean(np.abs(y - base))),
            "spearman": float(spearmanr(y, p).statistic), "pct_a_un_punto": float(100 * np.mean(np.abs(y - p) <= 1))}


In [ ]:
%%writefile caracteristicas.py
"""SPRINT 2 — Ingeniería de características: columnas nuevas y útiles sobre la tabla plana dataset_matricula.

Regla de oro contra la FUGA DE DATOS: todo lo que se APRENDE de los datos (las medias para estandarizar, los grupos
de k-means y el modelo NLP de la entrevista) se ajusta SOLO con el conjunto de ENTRENAMIENTO
(2024-1 a 2025-2) y después se aplica igual a validación, prueba, campaña y a cualquier aspirante nuevo (Sprint 6).
Si se ajustara con todos los datos, el modelo "vería" información del futuro y sus métricas serían optimistas.

Funciones principales:
  particion_temporal(periodo)                      -> ENTRENAMIENTO / VALIDACION / PRUEBA / CAMPANA
  ajustar_referencias(inscritos)                   -> lo aprendido del entrenamiento (un diccionario)
  construir_caracteristicas(inscritos, referencias) -> la misma tabla con las columnas nuevas
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import nlp_vocacion as nlp

NUEVAS = [
    "particion", "aplica_beca_saber11", "aplica_beca_mi_vocacion", "puntaje_vocacional_nlp",
    "segmento_rendimiento_icfes",
    "ahorro_beca_primer_semestre", "log_distancia", "zona_distancia", "rezago_saber11",
    "canal_agrupado", "estrato_grupo", "colegio_privado", "segmento_mercado",
]

# Descripción de cada columna nueva (diccionario del Sprint 2)
DICCIONARIO = {
    "particion": "Conjunto temporal: ENTRENAMIENTO (2024-1 a 2025-2), VALIDACION (2026-1), PRUEBA (2026-2), CAMPANA (2027-1)",
    "aplica_beca_saber11": "Sí/No: puntaje global Saber 11 ≥ 310 (requisito de la Beca Saber 11)",
    "aplica_beca_mi_vocacion": "Sí/No: presentó entrevista y su puntaje vocacional es ≥ 3 (requisito de Mi Vocación)",
    "puntaje_vocacional_nlp": "Puntaje vocacional de 1 a 5 que el modelo NLP lee en las palabras del aspirante (respuestas de la entrevista; TF-IDF + Ridge ajustado con el entrenamiento); vacío si no hubo entrevista",
    "segmento_rendimiento_icfes": "BAJO (< 250), MEDIO (250-309), ALTO (310-349: ya aplica a beca) o SUPERIOR (≥ 350)",
    "ahorro_beca_primer_semestre": "Cuánto le ahorra la beca en el 1er semestre (tarifa plena − precio neto), en COP",
    "log_distancia": "Logaritmo natural de la distancia por vías desde la dirección hasta el campus (S-44; reduce la cola larga)",
    "zona_distancia": "Distancia por vías desde la dirección: CERCA (≤ 10 km), MEDIA (10-20 km) o LEJOS (> 20 km)",
    "rezago_saber11": "INMEDIATO (presentó el Saber 11 el año anterior), UN_ANIO o DOS_O_MAS",
    "canal_agrupado": "EVENTO (Expo, Campus Day), RELACIONAL (referido, inmersión, convenio) o DIGITAL (web, redes, llamada)",
    "estrato_grupo": "BAJO (1-2), MEDIO (3) o ALTO (4-6)",
    "colegio_privado": "1 si el colegio es privado",
    "segmento_mercado": "Perfil del aspirante según k-means (segmentación de mercado), nombrado por sus rasgos principales",
}

ETIQUETAS_KMEANS = {   # (si el centro está por encima, si está por debajo) de la media del entrenamiento
    "estrato": ("estrato más alto", "estrato más bajo"),
    "log_distancia": ("viven lejos del campus", "viven cerca del campus"),
    "punt_global": ("buen puntaje ICFES", "puntaje ICFES bajo"),
    "edad_al_inscribirse": ("aspirantes mayores", "aspirantes más jóvenes"),
    "colegio_privado": ("colegio privado", "colegio público"),
    "colegio_promedio_saber11": ("colegio de alto nivel", "colegio de nivel bajo"),
}

# Nombres en español sencillo para tablas y gráficas (los nombres técnicos de las columnas no cambian)
NOMBRES_LEGIBLES = {
    "estrato": "Estrato promedio",
    "log_distancia": "Distancia por vías (km)",
    "distancia_vial_km": "Distancia por vías (km)",
    "punt_global": "Puntaje ICFES",
    "edad_al_inscribirse": "Edad (años)",
    "colegio_privado": "% de colegio privado",
    "colegio_promedio_saber11": "Nivel ICFES del colegio",
    "segmento_mercado": "Segmento de mercado",
    "segmento_rendimiento_icfes": "Puntaje ICFES",
    "aplica_beca_mi_vocacion": "¿Aplica a la beca Mi Vocación?",
    "aplica_beca_saber11": "¿Aplica a la beca Saber 11?",
    "canal_agrupado": "Canal por el que llegó",
    "zona_distancia": "Distancia al campus",
    "estrato_grupo": "Estrato",
    "rezago_saber11": "Tiempo desde el ICFES",
    "puntaje_vocacional_nlp": "Puntaje vocacional según sus palabras (NLP, 1 a 5)",
}

# El ICFES llama "oficial" al colegio público. Los datos y los modelos conservan el código OFICIAL; en todo texto que
# se muestra (explicaciones, gráficas, script de predicción) se dice "público".
NOMBRE_TIPO_COLEGIO = {"OFICIAL": "público", "PRIVADO": "privado", "TECNICO": "técnico"}


def tipo_colegio_codigo(valor: str) -> str:
    """Código de datos del tipo de colegio: acepta PUBLICO (o PÚBLICO) como sinónimo de OFICIAL."""
    v = str(valor).strip().upper().replace("Ú", "U").replace("É", "E")
    return "OFICIAL" if v == "PUBLICO" else v


def etiquetas_categorias(cfg: dict | None = None) -> dict:
    """Texto claro para los códigos de las columnas categóricas nuevas (p. ej. CERCA → "Cerca (hasta 10 km)")."""
    c = (cfg or configuracion())["caracteristicas"]
    z = c["zonas_distancia_km"]
    icfes = {b["segmento"]: (f"{b['segmento'].capitalize()} (menos de {b['hasta']})" if b["desde"] == 0 else
                             f"{b['segmento'].capitalize()} ({b['desde']} o más)" if b["hasta"] > 500 else
                             f"{b['segmento'].capitalize()} ({b['desde']} a {b['hasta'] - 1})") for b in c["bandas_icfes"]}
    return {
        "segmento_rendimiento_icfes": icfes,
        "zona_distancia": {"CERCA": f"Cerca (hasta {z['CERCA']} km)", "MEDIA": f"Media ({z['CERCA']} a {z['MEDIA']} km)",
                           "LEJOS": f"Lejos (más de {z['MEDIA']} km)"},
        "canal_agrupado": {"EVENTO": "Eventos (Expo, Campus Day)", "RELACIONAL": "Relacional (referido, convenio…)",
                           "DIGITAL": "Digital (web, redes, llamada)"},
        "estrato_grupo": {"BAJO": "Estratos 1 y 2", "MEDIO": "Estrato 3", "ALTO": "Estratos 4 a 6"},
        "rezago_saber11": {"INMEDIATO": "Sin pausa tras el colegio", "UN_ANIO": "1 año de pausa", "DOS_O_MAS": "2 años o más de pausa"},
        "aplica_beca_mi_vocacion": {"Sí": "Sí aplica", "No": "No aplica"},
        "aplica_beca_saber11": {"Sí": "Sí aplica", "No": "No aplica"},
    }


def tabla_segmentos(d: pd.DataFrame) -> pd.DataFrame:
    """Resumen de cada segmento de mercado con valores reales (no estandarizados), listo para leer o presentar."""
    m = d.assign(matriculado=d["matriculado"].astype(float))
    def resumen(g: pd.DataFrame) -> dict:
        return {"Inscritos": len(g), "% del total": len(g) / len(d) * 100,
                "% que se matriculó": g["matriculado"].mean() * 100, "Puntaje ICFES": g["punt_global"].mean(),
                "Distancia por vías (km)": g["distancia_vial_km"].mean(), "Estrato promedio": g["estrato"].mean(),
                "% de colegio público": (1 - g["colegio_privado"].mean()) * 100, "Edad (años)": g["edad_al_inscribirse"].mean()}
    t = pd.DataFrame({s: resumen(g) for s, g in m.groupby("segmento_mercado")}).T
    t.loc["Todos los inscritos"] = resumen(m)
    t.index.name = "Segmento de mercado"
    return t.round({"Inscritos": 0, "% del total": 1, "% que se matriculó": 1, "Puntaje ICFES": 0,
                    "Distancia por vías (km)": 1, "Estrato promedio": 1,
                    "% de colegio público": 0, "Edad (años)": 1}).astype({"Inscritos": int})


def configuracion() -> dict:
    """Parámetros del proyecto: config/reglas_negocio.yaml en el computador; en Colab, la copia del cuaderno."""
    try:
        from comun import cargar_config
        return cargar_config()
    except ImportError:
        return CONFIG_PROYECTO  # noqa: F821  (lo define una celda del cuaderno de Colab)


# ----------------------------------------------------------------------------- reglas (no se aprenden de datos)
def particion_temporal(periodo: pd.Series, cfg: dict | None = None) -> pd.Series:
    p = (cfg or configuracion())["particion"]
    mapa = {per: nombre.upper() for nombre in ("entrenamiento", "validacion", "prueba", "campana") for per in p[nombre]}
    return periodo.map(mapa)


def segmento_icfes(punt_global: pd.Series, cfg: dict | None = None) -> pd.Series:
    bandas = (cfg or configuracion())["caracteristicas"]["bandas_icfes"]
    cortes = [bandas[0]["desde"]] + [b["hasta"] for b in bandas]
    return pd.cut(punt_global, bins=cortes, right=False, labels=[b["segmento"] for b in bandas]).astype(str)


def _reglas(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    c = cfg["caracteristicas"]
    d = df.copy()
    d["particion"] = particion_temporal(d["periodo"], cfg)
    d["aplica_beca_saber11"] = np.where(d["punt_global"] >= cfg["becas"]["SABER_11"]["umbral_icfes"], "Sí", "No")
    d["aplica_beca_mi_vocacion"] = np.where(d["cumple_req_mi_vocacion"] == 1, "Sí", "No")
    d["segmento_rendimiento_icfes"] = segmento_icfes(d["punt_global"], cfg)
    # El precio que se usa es el neto del 1er semestre (valor_neto_primer_semestre, ya en dataset_matricula)
    d["ahorro_beca_primer_semestre"] = d["valor_semestre_bruto"] - d["valor_neto_primer_semestre"]
    vial = d["distancia_vial_km"].astype(float)                 # desde la dirección del formulario (S-44)
    d["log_distancia"] = np.log(np.maximum(vial, 0.5))           # piso de 0,5 km: un vecino del campus no da −∞
    z = c["zonas_distancia_km"]
    d["zona_distancia"] = np.select([vial <= z["CERCA"], vial <= z["MEDIA"]],
                                    ["CERCA", "MEDIA"], "LEJOS")
    d["rezago_saber11"] = np.select([d["anios_desde_saber11"] == 0, d["anios_desde_saber11"] == 1],
                                    ["INMEDIATO", "UN_ANIO"], "DOS_O_MAS")
    canal = {canal: grupo for grupo, canales in c["canal_agrupado"].items() for canal in canales}
    d["canal_agrupado"] = d["canal"].map(canal)
    d["estrato_grupo"] = d["estrato"].map(lambda e: c["estrato_grupo"][int(e)])
    d["colegio_privado"] = (d["colegio_naturaleza"] == "PRIVADO").astype(int)
    return d


# ----------------------------------------------------------------------------- lo que se aprende del entrenamiento
def ajustar_referencias(inscritos: pd.DataFrame, cfg: dict | None = None) -> dict:
    """Aprende, SOLO con las filas de ENTRENAMIENTO, todo lo que necesitan las columnas nuevas."""
    cfg = cfg or configuracion()
    ent = _reglas(inscritos, cfg)
    ent = ent[ent["particion"] == "ENTRENAMIENTO"]
    ref: dict = {"periodos_entrenamiento": sorted(ent["periodo"].unique()), "n_entrenamiento": len(ent)}

    # 1) Estandarización (z) del Saber 11 con la media y la desviación del entrenamiento (término cuadrático del modelo)
    ref["estandarizacion"] = {c: (float(ent[c].mean()), float(ent[c].std())) for c in ["punt_global"]}

    # 2) Segmentación de mercado con k-means: se prueba k = k_min..k_max; se elige el MENOR k cuya silueta esté a
    #    menos de tolerancia_silueta de la mejor (parsimonia: menos segmentos son más fáciles de usar para Mercadeo)
    km = cfg["caracteristicas"]["kmeans"]
    variables = km["variables"]
    medias, desv = ent[variables].mean(), ent[variables].std()
    Z = ((ent[variables] - medias) / desv).to_numpy(float)
    evaluacion = []
    for k in range(km["k_min"], km["k_max"] + 1):
        modelo = KMeans(n_clusters=k, n_init=20, random_state=km["semilla"]).fit(Z)
        evaluacion.append({"k": k, "inercia": float(modelo.inertia_),
                           "silueta": float(silhouette_score(Z, modelo.labels_, random_state=km["semilla"]))})
    tope = max(e["silueta"] for e in evaluacion) - km.get("tolerancia_silueta", 0.0)
    mejor = min((e for e in evaluacion if e["silueta"] >= tope), key=lambda e: e["k"])
    modelo = KMeans(n_clusters=mejor["k"], n_init=20, random_state=km["semilla"]).fit(Z)
    orden = np.argsort(-np.bincount(modelo.labels_))            # segmento 1 = el más grande
    centros = pd.DataFrame(modelo.cluster_centers_[orden], columns=variables)
    nombres = []
    for i, fila in centros.iterrows():
        rasgos = fila.abs().sort_values(ascending=False).index[:2]
        rasgos_txt = ", ".join(ETIQUETAS_KMEANS[v][0 if fila[v] > 0 else 1] for v in rasgos)
        nombres.append(f"S{i + 1}: " + rasgos_txt[0].upper() + rasgos_txt[1:])   # "S1: Colegio público, viven cerca…"
    ref["kmeans"] = {"modelo": modelo, "orden": orden.tolist(), "variables": variables, "medias": medias, "desv": desv,
                     "evaluacion": evaluacion, "k": mejor["k"], "nombres": nombres, "centros_z": centros}

    # 3) Modelo NLP de la entrevista: lee el puntaje vocacional en las respuestas abiertas (src/nlp_vocacion.py).
    #    Las filas de entrenamiento quedan con su predicción fuera de pliegue.
    ref["nlp"] = nlp.ajustar(ent, cfg)
    return ref


def construir_caracteristicas(inscritos: pd.DataFrame, ref: dict, cfg: dict | None = None) -> pd.DataFrame:
    """Devuelve la misma tabla con las columnas nuevas, aplicando lo aprendido del entrenamiento."""
    cfg = cfg or configuracion()
    d = _reglas(inscritos, cfg)
    km = ref["kmeans"]
    Z = ((d[km["variables"]] - km["medias"]) / km["desv"]).to_numpy(float)
    etiqueta = km["modelo"].predict(Z)
    posicion = {original: nuevo for nuevo, original in enumerate(km["orden"])}
    d["segmento_mercado"] = [km["nombres"][posicion[e]] for e in etiqueta]
    d["puntaje_vocacional_nlp"] = nlp.puntuar(ref["nlp"], d)
    # El texto de la entrevista es la entrada del NLP, no una columna del modelo: no pasa a la tabla del Sprint 2
    originales = [c for c in inscritos.columns if c not in nlp.SECCIONES]
    return d[originales + [c for c in NUEVAS if c not in originales]]


In [ ]:
%%writefile modelado.py
"""Modelos y métricas de la matrícula (Sprint 2: regresión logística · Sprint 3: comparación con Random Forest y XGBoost).

El primer modelo del proyecto (Sprint 2) es una REGRESIÓN LOGÍSTICA sin ajuste de hiperparámetros
(config/reglas_negocio.yaml -> modelado.sprint2_logistica): es el modelo base del proyecto. En el Sprint 3 se ajustan
sus hiperparámetros con la validación y se compara con Random Forest y XGBoost.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score, log_loss, precision_score,
                             recall_score, roc_auc_score)


def metricas(y, p, umbral: float = 0.5) -> dict:
    """Métricas de clasificación. 'error_pct' es el lenguaje del docente: % de inscritos mal clasificados."""
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    pred = (p >= umbral).astype(int)
    unica = len(np.unique(y)) > 1
    return {
        "n": int(len(y)), "umbral": float(umbral),
        "error_pct": float(100 * (pred != y).mean()),
        "exactitud": float((pred == y).mean()),
        "auc_roc": float(roc_auc_score(y, p)) if unica else float("nan"),
        "auc_pr": float(average_precision_score(y, p)) if unica else float("nan"),
        "brier": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, p, labels=[0, 1])),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "tasa_real": float(y.mean()), "tasa_predicha_media": float(p.mean()),
    }


def tabla_metricas(resultados: dict[str, dict]) -> pd.DataFrame:
    """{nombre_modelo: metricas} -> tabla ordenada para mostrar."""
    cols = ["n", "error_pct", "exactitud", "auc_roc", "auc_pr", "brier", "precision", "recall", "f1"]
    return pd.DataFrame(resultados).T[cols].astype(float).round(4)


# =============================================================================
# Modelos de Machine Learning (regresión logística desde el Sprint 2; Random Forest y XGBoost en el Sprint 3)
# =============================================================================
# Variables de entrada. Todas se conocen ANTES de la decisión que se predice (sin fuga de datos).
# No se usa colegio_promedio_saber11 (media ICFES del colegio): Admisiones no suele tener ese dato. Tampoco el promedio
# del colegio: dejó de ser una variable del análisis (sus notas quedan solo en las tablas del bachillerato).
# Sprint 2 (modelo base): la entrevista entra con el puntaje vocacional del entrevistador, sin el NLP.
VARIABLES_MATRICULA = {
    "numericas": ["punt_global", "icfes_al_cuadrado", "edad_al_inscribirse", "estrato",
                  "log_distancia", "dias_registro_a_inscripcion", "contactos_antes_de_inscribirse",
                  "anios_desde_saber11", "puntaje_vocacional", "pct_descuento_inicial",
                  "valor_neto_primer_semestre"],
    "binarias": ["entrevista_realizada", "apoyo_foes", "elegible_foes", "colegio_en_convenio", "captado_en_evento",
                 "colegio_privado", "semestre_2"],
    "categoricas": ["programa", "genero", "canal", "beca", "tipo_colegio", "regimen_salud", "segmento_mercado"],
}
# Sprint 3: las mismas variables + el puntaje vocacional que el modelo NLP lee en las palabras del aspirante
VARIABLES_MATRICULA_NLP = {**VARIABLES_MATRICULA,
                           "numericas": VARIABLES_MATRICULA["numericas"] + ["puntaje_vocacional_nlp"]}


def preparar_X(d: pd.DataFrame, referencias: dict, variables: dict) -> pd.DataFrame:
    """Columnas de entrada del modelo, con el tratamiento de nulos del Sprint 1 y dos derivadas simples."""
    X = d.copy()
    # 0 = sin entrevista (entrevista_realizada distingue "sin entrevista" de un puntaje bajo)
    for c in ("puntaje_vocacional", "puntaje_vocacional_nlp"):
        if c in X:
            X[c] = X[c].astype("Float64").fillna(0).astype(float)
    media, desv = referencias["estandarizacion"]["punt_global"]
    X["icfes_al_cuadrado"] = ((X["punt_global"] - media) / desv) ** 2       # captura la "U invertida" del ICFES
    X["semestre_2"] = X["periodo"].str.endswith("-2").astype(int)          # ingreso a mitad de año
    columnas = variables["numericas"] + variables["binarias"] + variables["categoricas"]
    X = X[columnas].copy()
    for c in variables["categoricas"]:
        X[c] = X[c].astype(str)
    for c in variables["numericas"] + variables["binarias"]:
        X[c] = X[c].astype(float)
    return X


def construir_pipeline(tipo: str, parametros: dict, variables: dict, semilla: int = 20261105):
    """Preprocesamiento + modelo en un solo objeto: se ajusta y se aplica igual en todos los conjuntos."""
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier

    categ = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    if tipo == "logistica":
        prep = ColumnTransformer([("num", StandardScaler(), variables["numericas"] + variables["binarias"]),
                                  ("cat", categ, variables["categoricas"])], verbose_feature_names_out=False)
        modelo = LogisticRegression(max_iter=5000, random_state=semilla, **parametros)   # saga baraja: semilla fija
    else:
        prep = ColumnTransformer([("num", "passthrough", variables["numericas"] + variables["binarias"]),
                                  ("cat", categ, variables["categoricas"])], verbose_feature_names_out=False)
        if tipo == "random_forest":
            modelo = RandomForestClassifier(n_estimators=500, random_state=semilla, n_jobs=-1, **parametros)
        elif tipo == "xgboost":
            modelo = XGBClassifier(random_state=semilla, n_jobs=1, eval_metric="logloss", tree_method="hist",
                                   subsample=0.8, colsample_bytree=0.8, **parametros)
        else:
            raise ValueError(tipo)
    return Pipeline([("preparacion", prep), ("modelo", modelo)])


def grilla(tipo: str, cfg: dict) -> list[dict]:
    """Combinaciones de hiperparámetros a probar (config/reglas_negocio.yaml -> modelado.grillas)."""
    from itertools import product
    g = cfg["modelado"]["grillas"][tipo]
    return [dict(zip(g, valores)) for valores in product(*g.values())]


def ajustar_temporal(tipo: str, cfg: dict, Xe, ye, Xv, yv, variables: dict) -> pd.DataFrame:
    """Prueba cada combinación entrenando con ENTRENAMIENTO y midiendo en VALIDACION (2026-1)."""
    filas = []
    for params in grilla(tipo, cfg):
        modelo = construir_pipeline(tipo, params, variables, cfg["modelado"]["semilla"]).fit(Xe, ye)
        m = metricas(yv, modelo.predict_proba(Xv)[:, 1])
        filas.append({"tipo": tipo, **{f"param_{k}": v for k, v in params.items()},
                      "log_loss_validacion": m["log_loss"], "auc_validacion": m["auc_roc"],
                      "brier_validacion": m["brier"], "error_pct_validacion": m["error_pct"], "parametros": params})
    return pd.DataFrame(filas).sort_values("log_loss_validacion").reset_index(drop=True)


def intervalo_bootstrap(y, p, funcion, n: int = 1000, semilla: int = 7, umbral: float = 0.5) -> tuple[float, float]:
    """IC 95% por remuestreo: se repite la medición en 1.000 muestras con reemplazo del conjunto de prueba."""
    y, p = np.asarray(y, dtype=int), np.asarray(p, dtype=float)
    rng = np.random.default_rng(semilla)
    valores = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        valores.append(funcion(y[i], p[i], umbral))
    return float(np.percentile(valores, 2.5)), float(np.percentile(valores, 97.5))


def diferencia_error(y, p_antes, p_despues, n: int = 1000, semilla: int = 7, umbral: float = 0.5) -> tuple[float, float]:
    """IC 95% de (error del modelo nuevo − error del anterior), en puntos, remuestreando los MISMOS inscritos.
    Si el intervalo incluye el 0, la diferencia puede ser azar: los dos modelos rinden igual."""
    y = np.asarray(y, dtype=int)
    falla_a = (np.asarray(p_antes) >= umbral) != y
    falla_b = (np.asarray(p_despues) >= umbral) != y
    rng = np.random.default_rng(semilla)
    dif = [100 * (falla_b[i].mean() - falla_a[i].mean()) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return float(np.percentile(dif, 2.5)), float(np.percentile(dif, 97.5))


def frase_docente(error_antes: float, error_despues: float, nombre: str, ic: tuple[float, float], n: int,
                  antes: str = "La regresión logística de la Semana 2", cuando: str = "inscritos de 2026-2") -> str:
    """La frase que pide el docente ("tenía un error del X%, este modelo del Y%"), sin afirmar una mejora que no hay."""
    num = lambda v: f"{v:.1f}".replace(".", ",")   # noqa: E731
    frase = (f"{antes} tenía un error del {num(error_antes)}%, mientras que el modelo de la Semana 3 ({nombre}) tiene "
             f"un error del {num(error_despues)}% en {n:,} {cuando}, que ningún modelo vio al entrenar.".replace(f"{n:,}", f"{n:,}".replace(",", ".")))
    dif = error_despues - error_antes
    if ic[1] < 0:
        return frase + f" Mejora {num(-dif)} puntos (IC 95% de la diferencia: {num(ic[0])} a {num(ic[1])})."
    if ic[0] > 0:
        return frase + f" Es {num(dif)} puntos peor (IC 95% de la diferencia: {num(ic[0])} a {num(ic[1])})."
    return (frase + f" La diferencia ({num(dif)} puntos) está dentro del margen del azar (IC 95%: {num(ic[0])} a "
            f"{num(ic[1])}): los dos modelos rinden igual.")


def nombres_salida(modelo) -> list[str]:
    """Nombres de las columnas que realmente ve el modelo (después del one-hot)."""
    return list(modelo.named_steps["preparacion"].get_feature_names_out())


In [ ]:
%%writefile flujo.py
"""Paso común de los sprints 3 a 6: cargar las tablas planas y construir las características del Sprint 2.

Así todos los sprints usan exactamente las mismas columnas, aprendidas solo con el entrenamiento.
"""
from __future__ import annotations

import pandas as pd

import caracteristicas as ca
import datos


def cargar_y_construir(fuente: str = "auto") -> dict:
    """Devuelve los inscritos con características, las referencias del Sprint 2, el origen de los datos y las
    respuestas de la entrevista por sección (la entrada del modelo NLP; no son columnas del modelo de matrícula)."""
    import nlp_vocacion as nv
    inscritos, origen = datos.cargar_inscritos(fuente)
    ref = ca.ajustar_referencias(inscritos)
    return {"inscritos": ca.construir_caracteristicas(inscritos, ref), "referencias": ref, "origen": origen,
            "entrevista": inscritos[["inscripcion_id", *nv.SECCIONES]]}


def conjunto(d: pd.DataFrame, *nombres: str) -> pd.DataFrame:
    """Filtra por partición temporal: conjunto(d, 'ENTRENAMIENTO', 'VALIDACION')."""
    return d[d["particion"].isin(nombres)]


def modelos_finales(f: dict, parametros: dict) -> dict:
    """Reentrena los modelos finales del Sprint 3 con los hiperparámetros ya elegidos (los guarda modelos/*.json).

    Sirve en Google Colab, donde no están los archivos .joblib: con la misma semilla el resultado es idéntico.
    parametros = {"matricula": {"tipo", "parametros"}, "xgboost": {"parametros"}}
    """
    import modelado as mo
    cfg = ca.configuracion()
    semilla = cfg["modelado"]["semilla"]
    d, ref = f["inscritos"], f["referencias"]
    hist = conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
    V, V2 = mo.VARIABLES_MATRICULA_NLP, mo.VARIABLES_MATRICULA          # Sprint 3 (con NLP) · Sprint 2 (sin NLP)
    Xh, yh = mo.preparar_X(hist, ref, V), hist["matriculado"].astype(int)
    pm = parametros["matricula"]
    return {
        "referencias": ref,
        "modelo_matricula": mo.construir_pipeline(pm["tipo"], pm["parametros"], V, semilla).fit(Xh, yh),
        "modelo_matricula_xgboost": mo.construir_pipeline("xgboost", parametros["xgboost"]["parametros"], V, semilla).fit(Xh, yh),
        # La regresión logística del Sprint 2 (sin ajuste de hiperparámetros ni NLP), para compararla en igualdad de condiciones
        "logistica_sprint2": mo.construir_pipeline("logistica", cfg["modelado"]["sprint2_logistica"], V2, semilla).fit(
            mo.preparar_X(hist, ref, V2), yh),
        "fondo_matricula": Xh.sample(300, random_state=1),
    }


In [ ]:
%%writefile graficas_sprint2.py
"""Gráficas del Sprint 2 en español sencillo. Las usan ejecutar_sprint2.py (las guarda en PNG) y el cuaderno (las muestra).

Cada función devuelve la figura de matplotlib. Los ejes y títulos usan nombres claros (NOMBRES_LEGIBLES) en vez de los
nombres técnicos de las columnas, y las barras llevan su valor escrito para leerlas sin mirar el eje.
"""
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import caracteristicas as ca

AZUL, NARANJA, GRIS, VERDE = "#1B365D", "#E07B39", "#9AA5B1", "#2E7D5B"
COLORES_SEGMENTO = ["#791734", "#C9A227", "#E07B39", "#4A6FA5", "#2E7D5B"]      # S1 a S5
NOMBRE_CORTO = {"estrato": "estrato", "log_distancia": "distancia al campus", "punt_global": "puntaje ICFES",
                "edad_al_inscribirse": "edad",
                "colegio_privado": "colegio privado", "colegio_promedio_saber11": "nivel del colegio"}
ORDEN = {"segmento_rendimiento_icfes": ["BAJO", "MEDIO", "ALTO", "SUPERIOR"], "zona_distancia": ["CERCA", "MEDIA", "LEJOS"],
         "estrato_grupo": ["BAJO", "MEDIO", "ALTO"], "rezago_saber11": ["INMEDIATO", "UN_ANIO", "DOS_O_MAS"],
         "aplica_beca_mi_vocacion": ["No", "Sí"], "canal_agrupado": ["DIGITAL", "EVENTO", "RELACIONAL"]}


def eleccion_k(ref: dict) -> plt.Figure:
    km = ref["kmeans"]
    ev = pd.DataFrame(km["evaluacion"])
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
    ejes[0].plot(ev["k"], ev["inercia"], "o-", color=AZUL)
    ejes[0].set(title="Método del codo: con más grupos, cada grupo es más parejo",
                xlabel="número de segmentos (k)", ylabel="dispersión dentro de los grupos (inercia)")
    ejes[1].plot(ev["k"], ev["silueta"], "o-", color=NARANJA)
    ejes[1].axvline(km["k"], color=GRIS, ls="--", label=f"se eligieron {km['k']} segmentos")
    ejes[1].set(title="Silueta: qué tan separados quedan los grupos (más alto = mejor)",
                xlabel="número de segmentos (k)", ylabel="silueta media (de −1 a 1)")
    ejes[1].legend(fontsize=9)
    for ax in ejes:
        ax.set_xticks(ev["k"])
    fig.tight_layout()
    return fig


def mapa_segmentos(d: pd.DataFrame, ref: dict) -> plt.Figure:
    """Cómo se ven los segmentos de k-means. k-means trabaja con 6 variables a la vez; para dibujarlas en un plano se
    proyectan en sus 2 componentes principales (PCA con las medias y escalas del entrenamiento). Cada punto es un
    inscrito, su color es el segmento que le asignó k-means y los círculos grandes son los centros de cada segmento."""
    km = ref["kmeans"]
    Z = ((d[km["variables"]] - km["medias"]) / km["desv"]).to_numpy(float)
    ent = (d["particion"] == "ENTRENAMIENTO").to_numpy()
    media = Z[ent].mean(axis=0)
    _, s, vt = np.linalg.svd(Z[ent] - media, full_matrices=False)
    comp = vt[:2].copy()
    for i in range(2):                                   # la variable con más peso apunta a la derecha / arriba
        if comp[i][np.argmax(np.abs(comp[i]))] < 0:
            comp[i] = -comp[i]
    var = (s ** 2 / (s ** 2).sum())[:2]
    P = (Z - media) @ comp.T
    C = (km["centros_z"].to_numpy(float) - media) @ comp.T          # centros, en el mismo orden que los nombres
    fig, ax = plt.subplots(figsize=(13.5, 7.6))
    for i, nombre in enumerate(km["nombres"]):
        m = (d["segmento_mercado"] == nombre).to_numpy()
        ax.scatter(P[m, 0], P[m, 1], s=16, alpha=0.35, color=COLORES_SEGMENTO[i % 5], edgecolors="none",
                   label=f"{nombre} ({format(int(m.sum()), ',').replace(',', '.')} inscritos)")
    for i, nombre in enumerate(km["nombres"]):
        ax.scatter(*C[i], s=900, color=COLORES_SEGMENTO[i % 5], edgecolors="white", linewidths=3, zorder=5)
        ax.annotate(nombre.split(":")[0], C[i], ha="center", va="center", color="white", fontsize=13,
                    fontweight="bold", zorder=6)

    def eje(i: int) -> str:
        cargas = pd.Series(comp[i], index=km["variables"])
        top = cargas.abs().sort_values(ascending=False).index[:2]
        partes = [("más " if cargas[v] > 0 else "menos ") + NOMBRE_CORTO.get(v, v) for v in top]
        return f"Dimensión {i + 1} ({var[i] * 100:.0f}% de la variación) → {' y '.join(partes)}"

    ax.set(xlabel=eje(0), ylabel=eje(1),
           title="Así se ven los segmentos de k-means: cada punto es un inscrito; el círculo grande, el centro del grupo")
    ley = ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=10, markerscale=2.2)
    for h in ley.legend_handles:
        h.set_alpha(1)
    ax.axhline(0, color=GRIS, lw=0.6, ls=":")
    ax.axvline(0, color=GRIS, lw=0.6, ls=":")
    fig.tight_layout()
    return fig


def perfil_segmentos(d: pd.DataFrame) -> plt.Figure:
    """Tabla de colores: cada celda muestra el valor REAL promedio del segmento; el color dice si está por encima
    (rojo) o por debajo (azul) del promedio de todos los inscritos."""
    t = ca.tabla_segmentos(d)
    cols = ["Puntaje ICFES", "Distancia por vías (km)", "Estrato promedio",
            "% de colegio público", "Edad (años)", "% que se matriculó"]
    valores = t[cols]
    seg, total = valores.drop(index="Todos los inscritos"), valores.loc["Todos los inscritos"]
    rel = (seg - total) / seg.std().replace(0, 1)
    fig, ax = plt.subplots(figsize=(13, 0.75 * len(valores) + 1.8))
    colores = np.vstack([rel.clip(-1.5, 1.5).to_numpy(float), np.zeros((1, len(cols)))])
    ax.imshow(colores, cmap="coolwarm", vmin=-2.2, vmax=2.2, aspect="auto")
    ax.set_xticks(range(len(cols)), [c.replace(" (", "\n(") for c in cols], fontsize=9)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(valores)), valores.index, fontsize=9)
    for (i, j), v in np.ndenumerate(valores.to_numpy(float)):
        texto = f"{v:.0f}%" if cols[j].startswith("%") else f"{v:.2f}" if "Notas" in cols[j] else \
            f"{v:.0f}" if cols[j] == "Puntaje ICFES" else f"{v:.1f}"
        ax.text(j, i, texto, ha="center", va="center", fontsize=10,
                fontweight="bold" if i == len(valores) - 1 else "normal")
    ax.axhline(len(valores) - 1.5, color="black", lw=1)
    ax.set_title("¿Cómo es cada segmento? Promedio de cada grupo  ·  rojo = más alto que el promedio · azul = más bajo",
                 fontsize=11, pad=12)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    fig.tight_layout()
    return fig


def tamano_y_matricula(d: pd.DataFrame) -> plt.Figure:
    t = ca.tabla_segmentos(d).drop(index="Todos los inscritos").iloc[::-1]
    tasa_total = d["matriculado"].astype(float).mean() * 100
    fig, ejes = plt.subplots(1, 2, figsize=(13, 0.6 * len(t) + 1.6), sharey=True)
    b = ejes[0].barh(t.index, t["Inscritos"], color=AZUL)
    ejes[0].bar_label(b, labels=[f"{n:,}  ({p:.0f}%)".replace(",", ".") for n, p in zip(t["Inscritos"], t["% del total"])],
                      padding=3, fontsize=9)
    ejes[0].set(title="¿Cuántos inscritos hay en cada segmento?", xlabel="inscritos")
    ejes[0].set_xlim(0, t["Inscritos"].max() * 1.3)
    b = ejes[1].barh(t.index, t["% que se matriculó"], color=[VERDE if v >= tasa_total else NARANJA for v in t["% que se matriculó"]])
    ejes[1].bar_label(b, fmt="%.0f%%", padding=3, fontsize=9)
    ejes[1].axvline(tasa_total, color=GRIS, ls="--", label=f"todos los inscritos: {tasa_total:.0f}%")
    ejes[1].set(title="¿Qué porcentaje se matriculó?", xlabel="% que pagó la matrícula", xlim=(0, 70))
    ejes[1].legend(fontsize=9, loc="lower right")
    fig.tight_layout()
    return fig


def tabla_matricula_por_grupo(ent: pd.DataFrame, columnas: list[str], cfg: dict | None = None) -> pd.DataFrame:
    """Tabla larga: para cada característica y cada grupo, cuántos inscritos hay y qué % se matriculó."""
    etiq = ca.etiquetas_categorias(cfg)
    filas = []
    for c in columnas:
        g = ent.assign(m=ent["matriculado"].astype(float)).groupby(c)["m"].agg(["size", "mean"])
        g = g.reindex([o for o in ORDEN.get(c, g.index) if o in g.index])
        for codigo, (n, tasa) in g.iterrows():
            filas.append({"Característica": ca.NOMBRES_LEGIBLES.get(c, c), "Grupo": etiq.get(c, {}).get(codigo, codigo),
                          "Inscritos": int(n), "% que se matriculó": round(tasa * 100, 1)})
    return pd.DataFrame(filas)


def matricula_por_grupo(ent: pd.DataFrame, columnas: list[str], cfg: dict | None = None) -> plt.Figure:
    t = tabla_matricula_por_grupo(ent, columnas, cfg)
    tasa_total = ent["matriculado"].astype(float).mean() * 100
    filas = int(np.ceil(len(columnas) / 3))
    fig, ejes = plt.subplots(filas, 3, figsize=(16, 3.6 * filas))
    for ax, c in zip(ejes.flat, columnas):
        g = t[t["Característica"] == ca.NOMBRES_LEGIBLES.get(c, c)].iloc[::-1]
        b = ax.barh(g["Grupo"], g["% que se matriculó"],
                    color=[VERDE if v >= tasa_total else NARANJA for v in g["% que se matriculó"]])
        ax.bar_label(b, labels=[f"{v:.0f}%  (n = {n})" for v, n in zip(g["% que se matriculó"], g["Inscritos"])],
                     padding=3, fontsize=8)
        ax.axvline(tasa_total, color=GRIS, ls="--")
        ax.set_xlim(0, 85)
        ax.set_title(ca.NOMBRES_LEGIBLES.get(c, c), fontsize=11)
        ax.set_xlabel("% que se matriculó")
    for ax in list(ejes.flat)[len(columnas):]:
        ax.axis("off")
    fig.suptitle(f"¿Qué porcentaje se matricula en cada grupo? · inscritos de entrenamiento ({len(ent):,}) · verde = más "
                 f"que el promedio ({tasa_total:.0f}%, línea gris)".replace(",", "."), fontsize=12)
    fig.tight_layout()
    return fig


def palabras_nlp(claves: pd.DataFrame) -> plt.Figure:
    """Las palabras que más suben (verde) y más bajan (naranja) el puntaje vocacional en cada sección de la entrevista.
    claves: salida de nlp_vocacion.palabras_clave."""
    secciones = list(dict.fromkeys(claves["seccion"]))
    fig, ejes = plt.subplots(1, len(secciones), figsize=(15, 5.2))
    for ax, s in zip(ejes, secciones):
        c = claves[claves["seccion"] == s].sort_values("peso")
        ax.barh(c["termino"], c["peso"], color=[VERDE if p > 0 else NARANJA for p in c["peso"]])
        ax.axvline(0, color=GRIS, lw=0.8)
        ax.set(title=s, xlabel="peso en el puntaje (+ sube, − baja)")
        ax.tick_params(axis="y", labelsize=9)
    fig.suptitle("Las palabras que determinan la vocación según el modelo NLP (TF-IDF + Ridge, ajustado con el entrenamiento)",
                 fontsize=12)
    fig.tight_layout()
    return fig


def puntaje_nlp(d: pd.DataFrame, conjunto: str = "VALIDACION") -> plt.Figure:
    """Izquierda: el puntaje NLP frente al del entrevistador. Derecha: % que se matricula por quintil del puntaje NLP."""
    con = d[(d["particion"] == conjunto) & d["puntaje_vocacional_nlp"].notna()]
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4.4))
    grupos = [con.loc[con["puntaje_vocacional"] == k, "puntaje_vocacional_nlp"] for k in range(1, 6)]
    ejes[0].boxplot(grupos, patch_artist=True, boxprops={"facecolor": "#DCE3EC"}, medianprops={"color": AZUL, "lw": 2})
    ejes[0].set_xticks(range(1, 6), [str(k) for k in range(1, 6)])     # sin tick_labels: sirve con cualquier matplotlib
    ejes[0].plot(range(1, 6), range(1, 6), ls="--", color=GRIS, lw=1, label="lectura perfecta")
    ejes[0].set(title="¿Las palabras dicen lo mismo que el entrevistador?", xlabel="puntaje del entrevistador (1 a 5)",
                ylabel="puntaje NLP según sus palabras")
    ejes[0].legend(fontsize=9)
    quintil = pd.qcut(con["puntaje_vocacional_nlp"], 5, labels=["Q1 (más bajo)", "Q2", "Q3", "Q4", "Q5 (más alto)"])
    tasa = con["matriculado"].astype(float).groupby(quintil, observed=True).mean() * 100
    sin = d.loc[(d["particion"] == conjunto) & (d["entrevista_realizada"] == 0), "matriculado"].astype(float).mean() * 100
    barras = ejes[1].bar(list(tasa.index.astype(str)) + ["sin entrevista"], list(tasa) + [sin],
                         color=[AZUL] * len(tasa) + [GRIS])
    ejes[1].bar_label(barras, fmt="%.0f%%", fontsize=9)
    ejes[1].set(title="¿Qué porcentaje se matricula según el puntaje NLP?", ylabel="% que se matriculó",
                ylim=(0, max(list(tasa) + [sin]) * 1.2))
    ejes[1].tick_params(axis="x", labelsize=8)
    nombre = {"VALIDACION": "validación 2026-1", "ENTRENAMIENTO": "entrenamiento"}.get(conjunto, conjunto.lower())
    fig.suptitle(f"Puntaje vocacional NLP · {nombre} ({len(con):,} entrevistas)".replace(",", "."), fontsize=11)
    fig.tight_layout()
    return fig


def error_logistica(metr: pd.DataFrame) -> plt.Figure:
    """metr: índice = conjunto (VALIDACION, PRUEBA) con las columnas error_pct y auc_roc de la regresión logística."""
    conjuntos = {"VALIDACION": "Validación (2026-1)", "PRUEBA": "Prueba (2026-2)"}
    m = metr.rename(index=conjuntos).loc[list(conjuntos.values())]
    fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, col, titulo, fmt in [(ejes[0], "error_pct", "¿Cuántas veces se equivoca? (más bajo = mejor)", "{:.1f}%"),
                                 (ejes[1], "auc_roc", "¿Qué tan bien ordena? AUC (0,5 = azar; 1 = perfecto)", "{:.3f}")]:
        barras = ax.bar(m.index, m[col], color=[AZUL, NARANJA])
        ax.bar_label(barras, labels=[fmt.format(v) for v in m[col]], fontsize=9)
        ax.set(title=titulo, ylim=(0, m[col].max() * 1.25))
    ejes[0].set_ylabel("% de inscritos mal clasificados")
    fig.suptitle("Regresión logística del Sprint 2", fontsize=11)
    fig.tight_layout()
    return fig


In [ ]:
%%writefile explicabilidad.py
"""SPRINT 4 — Machine Learning explicable: ¿por qué el modelo toma sus decisiones?

SHAP (Lundberg y Lee, 2017) reparte la predicción de cada inscrito entre sus variables, como un "pago justo" en un
juego cooperativo (valores de Shapley). Para un inscrito:

    log-odds predicho = valor base (promedio) + suma de las contribuciones SHAP de sus variables

Como las contribuciones SE SUMAN, se pueden agrupar variables que miden la misma idea (por ejemplo, la beca, el
descuento y el valor neto forman "Beca y precio"). Eso hace la explicación legible para las directivas.

LIME (Ribeiro et al., 2016) es otra mirada: ajusta un modelo lineal sencillo alrededor de UN inscrito, con datos
simulados parecidos a él. Si SHAP y LIME señalan los mismos factores, la explicación es más confiable.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from caracteristicas import NOMBRE_TIPO_COLEGIO

# Variables agrupadas en conceptos de negocio (contribuciones SHAP sumadas)
GRUPOS_MATRICULA = {
    "Beca y precio": ["beca", "pct_descuento_inicial", "valor_neto_primer_semestre"],
    "Puntaje Saber 11": ["punt_global", "icfes_al_cuadrado"],
    "Distancia al campus": ["log_distancia"],
    "Canal de captación": ["canal", "captado_en_evento"],
    # La entrevista es UN concepto: si la presentó, el puntaje del entrevistador y el que se lee en sus palabras (NLP).
    # Por separado se compensan (con las mismas palabras, un puntaje alto del entrevistador deja menos intención por
    # explicar, supuesto S-49): sumados dicen cuánto mueve la entrevista.
    "Entrevista vocacional": ["entrevista_realizada", "puntaje_vocacional", "puntaje_vocacional_nlp"],
    "Apoyo FOES municipal": ["apoyo_foes", "elegible_foes"],
    "Estrato y régimen de salud": ["estrato", "regimen_salud"],
    "Programa": ["programa"],
    "Ingreso a mitad de año": ["semestre_2"],
    "Colegio de procedencia": ["tipo_colegio", "colegio_privado", "colegio_en_convenio"],
    "Contacto con asesores": ["contactos_antes_de_inscribirse", "dias_registro_a_inscripcion"],
    "Edad y años desde el Saber 11": ["edad_al_inscribirse", "anios_desde_saber11"],
    "Segmento de mercado": ["segmento_mercado"],
    "Género": ["genero"],
}

def _variable_de(columna: str, variables: dict) -> str:
    """Nombre original de una columna transformada (canal_REFERIDO -> canal)."""
    if columna in variables["numericas"] + variables["binarias"]:
        return columna
    return next(c for c in variables["categoricas"] if columna.startswith(c + "_"))


def contribuciones(modelo, X_fondo: pd.DataFrame, X: pd.DataFrame, variables: dict) -> tuple[pd.DataFrame, float]:
    """Valores SHAP en log-odds, ya agregados por variable ORIGINAL. Devuelve (tabla n × variables, valor base)."""
    import shap
    prep, clf = modelo.named_steps["preparacion"], modelo.named_steps["modelo"]
    nombres = list(prep.get_feature_names_out())
    T_fondo, T = prep.transform(X_fondo), prep.transform(X)
    if hasattr(clf, "coef_"):            # regresión logística: SHAP lineal (exacto)
        explicador = shap.LinearExplainer(clf, shap.maskers.Independent(T_fondo, max_samples=len(T_fondo)))
    else:                                # árboles (Random Forest, XGBoost): TreeSHAP con 100 casos de referencia
        explicador = shap.TreeExplainer(clf, data=T_fondo[:100], model_output="raw", feature_perturbation="interventional")
    exp = explicador(T)
    valores = exp.values[..., 1] if exp.values.ndim == 3 else exp.values
    base = exp.base_values[..., 1] if np.ndim(exp.base_values) == 2 else exp.base_values
    tabla = pd.DataFrame(valores, columns=nombres, index=X.index)
    tabla = tabla.T.groupby([_variable_de(c, variables) for c in nombres]).sum().T
    return tabla[[c for c in X.columns if c in tabla.columns]], float(np.mean(base))


def agrupar(contrib: pd.DataFrame, grupos: dict) -> pd.DataFrame:
    """Suma las contribuciones de las variables de cada concepto (SHAP es aditivo)."""
    return pd.DataFrame({g: contrib[[v for v in vs if v in contrib]].sum(axis=1) for g, vs in grupos.items()})


def importancia(contrib: pd.DataFrame) -> pd.Series:
    """Importancia global = promedio del valor absoluto de la contribución (cuánto mueve, en cualquier sentido)."""
    return contrib.abs().mean().sort_values(ascending=False)


def contexto(fila: pd.Series, grupo: str) -> str:
    """Dato concreto que acompaña a cada factor en la explicación en español."""
    f = fila
    textos = {
        "Beca y precio": lambda: f"beca {str(f['beca']).replace('_', ' ').title()}, descuento {float(f['pct_descuento_inicial']):.0%}",
        "Puntaje Saber 11": lambda: f"puntaje global {int(f['punt_global'])}",
        "Distancia al campus": lambda: f"vive a {float(np.exp(f['log_distancia'])):.0f} km por vía",
        "Canal de captación": lambda: f"llegó por {str(f['canal']).replace('_', ' ').lower()}",
        "Entrevista vocacional": lambda: (f"puntaje vocacional {int(f['puntaje_vocacional'])}; según sus palabras {float(f['puntaje_vocacional_nlp']):.1f}"
                                          if f["entrevista_realizada"] == 1 else "sin entrevista"),
        "Apoyo FOES municipal": lambda: "con apoyo del fondo municipal" if f.get("apoyo_foes", 0) == 1 else "sin apoyo del fondo municipal",
        "Estrato y régimen de salud": lambda: f"estrato {int(f['estrato'])}",
        "Programa": lambda: f"programa {f['programa']}",
        "Ingreso a mitad de año": lambda: "ingresa en el segundo periodo del año" if f["semestre_2"] == 1 else "ingresa en el primer periodo del año",
        "Colegio de procedencia": lambda: f"colegio {NOMBRE_TIPO_COLEGIO.get(str(f['tipo_colegio']), str(f['tipo_colegio']).lower())}",
        "Contacto con asesores": lambda: f"{int(f['contactos_antes_de_inscribirse'])} contactos antes de inscribirse",
        "Edad y años desde el Saber 11": lambda: f"{int(f['edad_al_inscribirse'])} años",
        "Segmento de mercado": lambda: str(f["segmento_mercado"]),
        "Género": lambda: f"género {f['genero']}",
    }
    try:
        return textos[grupo]()
    except (KeyError, ValueError, TypeError):
        return ""


def top_factores(por_grupo: pd.DataFrame, X: pd.DataFrame, ids: pd.Series, k: int = 3, que: str = "matricularse") -> pd.DataFrame:
    """Bloque 4 del dashboard: los k conceptos que más mueven la predicción de cada inscrito, en español."""
    filas = []
    for (idx, contrib), iid in zip(por_grupo.iterrows(), ids):
        for orden, (grupo, valor) in enumerate(contrib.reindex(contrib.abs().sort_values(ascending=False).index)[:k].items(), 1):
            sube = valor > 0
            dato = contexto(X.loc[idx], grupo)
            filas.append({"inscripcion_id": int(iid), "orden": orden, "factor": grupo,
                          "direccion": "SUBE" if sube else "BAJA", "contribucion_log_odds": round(float(valor), 4),
                          "texto": f"{grupo}{f' ({dato})' if dato else ''}: {'aumenta' if sube else 'reduce'} la probabilidad de {que}."})
    return pd.DataFrame(filas)


def explicar_lime(modelo, X_ref: pd.DataFrame, fila: pd.DataFrame, variables: dict, n_variables: int = 8,
                  semilla: int = 11) -> list[tuple[str, float]]:
    """LIME: modelo lineal local alrededor de un inscrito. Las categorías se codifican como números para LIME."""
    from lime.lime_tabular import LimeTabularExplainer
    cols = list(X_ref.columns)
    cats = [cols.index(c) for c in variables["categoricas"]]
    niveles = {cols[i]: sorted(X_ref[cols[i]].unique()) for i in cats}

    def codificar(df: pd.DataFrame) -> np.ndarray:
        A = df.copy()
        for c, nv in niveles.items():
            A[c] = A[c].map({v: k for k, v in enumerate(nv)}).fillna(0)
        return A.to_numpy(float)

    def decodificar(A: np.ndarray) -> pd.DataFrame:
        df = pd.DataFrame(A, columns=cols)
        for c, nv in niveles.items():
            df[c] = [nv[int(round(min(max(v, 0), len(nv) - 1)))] for v in df[c]]
        for c in variables["numericas"] + variables["binarias"]:
            df[c] = df[c].astype(float)
        return df

    explicador = LimeTabularExplainer(codificar(X_ref), feature_names=cols, categorical_features=cats,
                                      categorical_names={i: niveles[cols[i]] for i in cats}, class_names=["no", "sí"],
                                      discretize_continuous=True, random_state=semilla, mode="classification")
    exp = explicador.explain_instance(codificar(fila)[0], lambda A: modelo.predict_proba(decodificar(A)),
                                      num_features=n_variables, num_samples=3000)
    return exp.as_list()


def importancia_permutacion(modelo, X: pd.DataFrame, y, grupos: dict, repeticiones: int = 10,
                            semilla: int = 5) -> pd.DataFrame:
    """Chequeo independiente de SHAP: se DESORDENA un concepto (todas sus columnas a la vez) y se mide cuánto cae el
    AUC. Si cae mucho, el modelo dependía de ese concepto. No usa los valores SHAP: es otra forma de verlo."""
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(semilla)
    auc_base = roc_auc_score(y, modelo.predict_proba(X)[:, 1])
    filas = []
    for grupo, cols in grupos.items():
        cols = [c for c in cols if c in X]
        caidas = []
        for _ in range(repeticiones):
            Xp = X.copy()
            orden = rng.permutation(len(X))
            Xp[cols] = X[cols].to_numpy()[orden]
            caidas.append(auc_base - roc_auc_score(y, modelo.predict_proba(Xp)[:, 1]))
        filas.append({"concepto": grupo, "caida_auc": float(np.mean(caidas)), "de_caida": float(np.std(caidas))})
    return pd.DataFrame(filas).sort_values("caida_auc", ascending=False).reset_index(drop=True)


def chequeos_coherencia(X: pd.DataFrame, contrib: pd.DataFrame, por_grupo: pd.DataFrame, prob: np.ndarray,
                        y_real: pd.Series | None = None) -> pd.DataFrame:
    """¿El modelo se comporta como se espera? Cada fila es una afirmación verificable con los datos."""
    from scipy.stats import spearmanr
    saber = contrib[["punt_global", "icfes_al_cuadrado"]].sum(axis=1)
    banda = pd.cut(X["punt_global"], [0, 230, 290, 330, 600], labels=["< 230", "230-289", "290-329", "≥ 330"])
    por_banda = saber.groupby(banda, observed=True).mean()
    # El concepto completo (beca + descuento + valor neto): con penalización L1 el modelo puede dejar en cero las
    # columnas de "beca" porque el descuento ya dice lo mismo, y la variable sola no mostraría el efecto
    beca = por_grupo["Beca y precio"].groupby(X["beca"]).mean()
    imp = importancia(por_grupo)
    entrevista = por_grupo["Entrevista vocacional"]
    con = X["entrevista_realizada"] == 1
    filas = [
        ("Más descuento → más probabilidad de matrícula",
         spearmanr(X["pct_descuento_inicial"], contrib["pct_descuento_inicial"] + contrib["valor_neto_primer_semestre"]).statistic > 0,
         "correlación entre descuento y su contribución SHAP (con el valor neto) > 0"),
        ("Las becas pesan a favor frente a no tener beca",
         bool(beca.get("SABER_11", 0) > beca.get("NINGUNA", 0) and beca.get("MI_VOCACION", 0) > beca.get("NINGUNA", 0)),
         "contribución media de la beca: " + ", ".join(f"{k} {v:+.2f}" for k, v in beca.items())),
        ("Más distancia → menos probabilidad",
         spearmanr(X["log_distancia"], contrib["log_distancia"]).statistic < 0, "correlación entre distancia y su contribución < 0"),
        ("Saber 11 en U invertida: los puntajes muy altos tienen más opciones y se matriculan menos",
         bool(por_banda.iloc[-1] < por_banda.iloc[1:3].max()),
         "contribución media por banda: " + ", ".join(f"{k} {v:+.2f}" for k, v in por_banda.items())),
        ("La entrevista vocacional pesa a favor",
         bool(entrevista[con].mean() > entrevista[~con].mean()),
         f"contribución media del concepto con entrevista {entrevista[con].mean():+.2f} vs. sin entrevista {entrevista[~con].mean():+.2f}"),
        ("Más vocación en sus palabras (puntaje NLP) → más probabilidad",
         bool(spearmanr(X.loc[con, "puntaje_vocacional_nlp"], contrib.loc[con, "puntaje_vocacional_nlp"]).statistic > 0),
         "correlación entre el puntaje NLP y su contribución, entre los entrevistados, > 0"),
        ("El género NO domina las decisiones",
         bool(list(imp.index).index("Género") >= len(imp) // 2 and imp["Género"] < 0.15 * imp.iloc[0]),
         f"puesto {list(imp.index).index('Género') + 1} de {len(imp)}; importancia {imp['Género']:.3f} vs. {imp.iloc[0]:.3f} del primero"),
    ]
    if y_real is not None:
        tasas = pd.DataFrame({"predicha": pd.Series(prob, index=X.index), "real": y_real.astype(float)}).groupby(X["genero"]).mean()
        brecha_pred = abs(tasas.loc["F", "predicha"] - tasas.loc["M", "predicha"])
        brecha_real = abs(tasas.loc["F", "real"] - tasas.loc["M", "real"])
        filas.append(("Paridad por género: el modelo no crea una brecha que no exista en los datos",
                      bool(brecha_pred <= brecha_real + 0.03),
                      f"brecha predicha F-M {brecha_pred:.3f} vs. brecha real {brecha_real:.3f}"))
    return pd.DataFrame(filas, columns=["chequeo", "cumple", "evidencia"])


def comparar_con_generador(X: pd.DataFrame, contrib: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Los datos son sintéticos: se conocen los efectos VERDADEROS (config: generador.matricula).
    ¿La explicación SHAP recupera la dirección de cada efecto real? Las variables sin efecto real deberían pesar poco."""
    from scipy.stats import spearmanr
    b = cfg["generador"]["matricula"]
    canal_shap = (contrib["canal"] + contrib["captado_en_evento"]).groupby(X["canal"]).mean()
    canal_real = pd.Series(b["beta_canal"]).reindex(canal_shap.index)
    estr_shap = contrib["estrato"].groupby(X["estrato"]).mean()
    estr_real = pd.Series({int(k): v for k, v in b["beta_estrato"].items()}).reindex(estr_shap.index)
    sentido = lambda v: "positivo" if v > 0 else "negativo"   # noqa: E731
    filas = [
        ("Descuento (beca)", f"positivo (β = {b['beta_descuento']})",
         sentido(spearmanr(X["pct_descuento_inicial"], contrib["pct_descuento_inicial"] + contrib["valor_neto_primer_semestre"] + contrib["beca"]).statistic)),
        ("Distancia al campus", f"negativo (β = {b['beta_log_distancia']})", sentido(spearmanr(X["log_distancia"], contrib["log_distancia"]).statistic)),
        ("Saber 11 (término cuadrático)", f"U invertida (β = {b['beta_icfes_cuadratico']})",
         "U invertida" if contrib["icfes_al_cuadrado"].corr(X["icfes_al_cuadrado"]) < 0 else "no recupera la U"),
        ("Entrevista vocacional", f"positivo (β = {b['beta_entrevista']})",
         sentido(contrib[["entrevista_realizada", "puntaje_vocacional", "puntaje_vocacional_nlp"]].sum(axis=1).corr(X["entrevista_realizada"]))),
        ("Apoyo FOES", f"positivo (β = {b['beta_foes']})", sentido(contrib["apoyo_foes"].corr(X["apoyo_foes"]))),
        ("Programa SSIA", f"positivo (β = {b['beta_programa_ssia']})",
         sentido(contrib["programa"][X["programa"] == "SSIA"].mean() - contrib["programa"][X["programa"] == "AD"].mean())),
        ("Ingreso a mitad de año", f"negativo (β = {b['beta_periodo_2']})", sentido(contrib["semestre_2"].corr(X["semestre_2"]))),
        ("Canal (orden de los 8 canales)", "inmersión > referido > campus day/convenio > web > redes > llamada > expo",
         f"correlación de Spearman con el orden real = {spearmanr(canal_shap, canal_real).statistic:.2f}"),
        ("Estrato (orden de los 6 estratos)", "estrato 1 el más bajo; 4 el más alto",
         f"correlación de Spearman con el orden real = {spearmanr(estr_shap, estr_real).statistic:.2f}"),
        ("Género (sin efecto real)", "ninguno (β = 0)", f"importancia {contrib['genero'].abs().mean():.3f} log-odds"),
    ]
    tabla = pd.DataFrame(filas, columns=["efecto", "verdadero (generador)", "según SHAP"])
    tabla["coincide"] = [
        (r["según SHAP"] in r["verdadero (generador)"]) if r["según SHAP"] in ("positivo", "negativo", "U invertida")
        else (float(r["según SHAP"].split("= ")[-1]) > 0.5 if "Spearman" in r["según SHAP"]
              else float(r["según SHAP"].split()[1]) < 0.05)
        for _, r in tabla.iterrows()]
    return tabla


In [ ]:
%%writefile impacto.py
"""SPRINT 5 — Impacto en matrículas e ingreso del primer semestre (COP, con las tarifas oficiales).

Todas las cifras son VALOR ESPERADO: probabilidad × valor. Los dos supuestos (capacidad de seguimiento y efecto de la
llamada) están en config/reglas_negocio.yaml -> impacto, y cada uno se mueve ±50% en el análisis de sensibilidad.
No se usan costos: el proyecto no tiene datos de cuánto cuesta el seguimiento ni la inversión en los eventos.

Escenario 1 · Seguimiento priorizado. Los asesores no alcanzan a hacer seguimiento intensivo a todos los inscritos.
  El seguimiento ayuda poco a quien ya decidió (p cerca de 0 o de 1) y más al indeciso (p cerca de 0,5):
      efecto(p) = efecto_máximo · 4 · p · (1 − p)
  Se comparan tres formas de elegir a quién llamar: al azar, a los de mayor probabilidad y a los indecisos (modelo).
Escenario 2 · Expo Estudiantes. El mismo seguimiento aplicado a los inscritos que llegan de la feria: cuántas
  matrículas más se esperan; además, qué segmentos de mercado priorizar.
"""
from __future__ import annotations

import numpy as np
import pandas as pd


def efecto_seguimiento(p, efecto_maximo: float) -> np.ndarray:
    p = np.asarray(p, dtype=float)
    return efecto_maximo * 4 * p * (1 - p)


def escenario_seguimiento(prob: pd.Series, valor_anio: pd.Series, capacidad_pct: float, efecto_maximo: float) -> pd.DataFrame:
    """Matrículas adicionales esperadas y su valor (precio neto del 1er semestre) según a quién se le hace seguimiento."""
    p, v = np.asarray(prob, dtype=float), np.asarray(valor_anio, dtype=float)
    k = int(round(capacidad_pct * len(p)))
    efecto = efecto_seguimiento(p, efecto_maximo)
    estrategias = {
        "Al azar (sin modelo)": None,
        "Mayor probabilidad primero": np.argsort(-p)[:k],
        "Indecisos primero (modelo)": np.argsort(-efecto)[:k],
    }
    filas = []
    for nombre, idx in estrategias.items():
        if idx is None:      # valor esperado de elegir k al azar = k × promedio
            extra, ingreso = k * efecto.mean(), k * (efecto * v).mean()
        else:
            extra, ingreso = efecto[idx].sum(), (efecto[idx] * v[idx]).sum()
        filas.append({"estrategia": nombre, "inscritos_con_seguimiento": k, "matriculas_adicionales": extra,
                      "ingreso_primer_semestre_cop": ingreso})
    return pd.DataFrame(filas)


def sensibilidad(funcion, parametros: dict, variar: list[str], salida: str, factor: float = 0.5) -> pd.DataFrame:
    """Gráfico de tornado: se mueve cada supuesto -50% y +50% (los demás fijos) y se mide el resultado."""
    base = funcion(**parametros)[salida]
    filas = []
    for nombre in variar:
        bajo = funcion(**(parametros | {nombre: parametros[nombre] * (1 - factor)}))[salida]
        alto = funcion(**(parametros | {nombre: parametros[nombre] * (1 + factor)}))[salida]
        filas.append({"supuesto": nombre, "valor_base": parametros[nombre], "resultado_bajo": bajo,
                      "resultado_base": base, "resultado_alto": alto, "rango": abs(alto - bajo)})
    return pd.DataFrame(filas).sort_values("rango", ascending=False).reset_index(drop=True)


def acciones_por_segmento(d: pd.DataFrame, prob: pd.Series, cortes: dict) -> pd.DataFrame:
    """Tabla para Mercadeo: cada segmento de mercado con su propensión, valor esperado y acción sugerida."""
    t = d.assign(prob=np.asarray(prob, dtype=float), valor_esperado=np.asarray(prob, dtype=float) * d["valor_neto_primer_semestre"],
                 lejos=(d["zona_distancia"] == "LEJOS").astype(float), con_beca=(d["beca"] != "NINGUNA").astype(float),
                 indeciso=((np.asarray(prob) >= cortes["BAJA"]) & (np.asarray(prob) < cortes["ALTA"])).astype(float))
    g = t.groupby("segmento_mercado").agg(inscritos=("inscripcion_id", "size"), propension_media=("prob", "mean"),
                                         pct_indecisos=("indeciso", "mean"), pct_lejos=("lejos", "mean"),
                                         pct_con_beca=("con_beca", "mean"), valor_esperado_por_inscrito=("valor_esperado", "mean"))

    def accion(r) -> str:
        partes = []
        if r["propension_media"] >= cortes["ALTA"] - 0.1:
            partes.append("asegurar: proceso de matrícula ágil y recordatorio de las fechas de pago")
        if r["pct_indecisos"] >= 0.5:
            partes.append("prioridad de seguimiento del asesor (muchos indecisos)")
        if r["pct_lejos"] >= 0.4:
            partes.append("ofrecer el beneficio de transporte y revisar FOES de su municipio")
        if r["pct_con_beca"] < 0.4:
            partes.append("informar requisitos de Beca Saber 11 y Mi Vocación (entrevista)")
        return "; ".join(partes) or "contacto estándar"

    g["accion_sugerida"] = g.apply(accion, axis=1)
    return g.sort_values("valor_esperado_por_inscrito", ascending=False)


In [ ]:
import os
import sys
import warnings

sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
EN_COLAB = True
# Sin clave de Supabase, los datos se leen de GitHub: data/processed de esta rama del repositorio
os.environ["RAMA_DATOS_GITHUB"] = "Sprint-5"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, GRIS = "#1B365D", "#E07B39", "#9AA5B1"

In [ ]:
# Conexión segura del Sprint 1 (el mismo código de src/conexion_supabase.py)
from conexion_supabase import (TABLA, TIEMPO_LIMITE_S, crear_cliente, leer_dataset_matricula, obtener_credenciales,
                               probar_solo_lectura)

pd.set_option("display.max_columns", 60)        # el Sprint 1 muestra las 45 columnas del dataset plano

---
**COMIENZA EL SPRINT 1** · entrega del jueves 22 de octubre de 2026 · archivos del sprint en `notebooks/ · src/ · sql/ · docs/`

# Sprint 1 — Conexión segura con Supabase, dataset plano y EDA inicial
**Proyecto:** Modelo Predictivo de Segmentación de Mercado y Propensión a la Matrícula · Unicervantes
(Ingeniería en Analítica de Datos e Ingeniería en Sistemas de Software e IA).

Este cuaderno cubre los tres entregables que pidió el docente para el Sprint 1:

| # | Entregable | Sección |
|---|---|---|
| 1 | Conexión funcional entre Python y Supabase con buenas prácticas de seguridad | 1 |
| 2 | Un único dataset plano con las variables principales para predecir la matrícula | 2 |
| 3 | Inicio del EDA: estadística descriptiva, distribuciones y tratamiento preliminar de nulos | 3 |

> **Sobre los datos.** Los puntajes Saber 11, el colegio, el estrato, el género y la edad de cada aspirante son
> registros **reales** y anónimos del ICFES (datos abiertos). Nombres, proceso comercial, notas del colegio y
> resultado de matrícula son **sintéticos**, generados con reglas y supuestos documentados en el repositorio.

## 1. Conexión segura con Supabase
**Cómo se protegen las credenciales**
- La URL y la clave **no están en el código**: se leen del archivo `.env` (en el computador, excluido por
  `.gitignore`) o de los **Secretos de Colab**. El código nunca las imprime.
- Se usa la clave **publicable**, no la secreta. En la base, la seguridad a nivel de fila (RLS) le permite **solo
  leer** las tablas planas (`dataset_matricula` y `dataset_entrevista`); las 28 tablas transaccionales están cerradas
  para la API.
- Si la clave se filtrara, solo serviría para leer este dataset anonimizado: no permite escribir ni ver datos
  personales.

**Prueba:** se leen todas las filas (la API entrega máximo 1.000 por petición, por eso se lee en páginas) y se
intenta modificar la tabla con la misma clave. La base debe rechazarlo.

In [ ]:
REPOSITORIO, RAMA = "xyosho/Proyecto-Homologacion-ML", "Sprint-5"
ORIGEN_RESPALDO = f"copia en GitHub (data/processed de la rama {RAMA} del repositorio)"


def cargar_respaldo(tabla: str = "dataset_matricula") -> pd.DataFrame:
    """Sin clave de Supabase: la misma tabla, leída directamente de GitHub. No hay que subir ningún archivo."""
    return pd.read_csv(f"https://raw.githubusercontent.com/{REPOSITORIO}/{RAMA}/data/processed/{tabla}.csv")

In [ ]:
def motivo(error: Exception) -> str:
    """Explica el fallo sin mostrar nunca la URL ni la clave."""
    nombre = type(error).__name__
    if isinstance(error, RuntimeError):
        return str(error)
    if "Timeout" in nombre:
        return f"Supabase no respondió en {TIEMPO_LIMITE_S} s: revise la conexión a internet."
    if "Connect" in nombre:
        return "No se pudo llegar al servidor: revise SUPABASE_URL y la conexión a internet."
    codigo = getattr(error, "code", "")
    if codigo in ("PGRST301", "401") or "JWT" in str(getattr(error, "message", "")) or "API key" in str(error):
        return "La clave fue rechazada: revise que SUPABASE_KEY sea la clave publicable del proyecto."
    return f"{nombre}: revise la URL, la clave y la red."


cliente = None
try:
    print("Paso 1/3: leyendo las credenciales (en Colab, acepte el aviso de acceso a los secretos si aparece)…", flush=True)
    url, clave = obtener_credenciales()                      # de .env o de los Secretos; no se muestran
    print("Paso 2/3: conectando con la API de Supabase…", flush=True)
    cliente = crear_cliente(url, clave)
    df = leer_dataset_matricula(cliente)
    del url, clave
    print("Paso 3/3: probando que la clave NO pueda escribir…", flush=True)
    rechazada = probar_solo_lectura(cliente)
    ORIGEN = "Supabase (API de solo lectura)"
    print(f"\nConexión exitosa: {len(df):,} filas y {df.shape[1]} columnas leídas de la tabla '{TABLA}'.")
    print("Prueba de escritura con la misma clave:",
          "RECHAZADA por la base (permiso denegado). La clave solo puede leer." if rechazada
          else "NO fue rechazada: revisar las políticas RLS.")
except Exception as error:
    cliente = None
    print(f"\nNo se usó Supabase: {motivo(error)}")
    print(f"Se lee la misma tabla desde la {ORIGEN_RESPALDO} (no hay que subir ningún archivo).", flush=True)
    df, ORIGEN = cargar_respaldo(), ORIGEN_RESPALDO
    print(f"{len(df):,} filas y {df.shape[1]} columnas leídas.")
print("Origen de los datos:", ORIGEN)

## 2. Dataset plano para el modelo de matrícula
La base de datos tiene dos capas:
- **Capa transaccional (OLTP):** 28 tablas normalizadas que registran el proceso de admisión. Es la fuente.
- **Capa analítica (OLAP):** la tabla **`dataset_matricula`**, ya consolidada en Supabase. Tiene **una fila por
  inscrito** y una columna por variable. La **distancia al campus** viene calculada. Las notas del colegio no entran:
  dejaron de ser variables del análisis y quedan solo en la capa transaccional.
  Python no necesita unir tablas.

Roles de las columnas:
- **Control** (no son predictores): `inscripcion_id`, `periodo`, `conjunto`.
- **Objetivo:** `matriculado` (1 = pagó la matrícula del primer semestre).
- **Predictores:** el resto. Se excluyeron los datos personales y las variables con **fuga de datos** (por ejemplo,
  documentos entregados o forma de financiación, que solo se conocen al matricularse).

`conjunto` separa el histórico con resultado (`ENTRENAMIENTO`, 2024-1 a 2026-2) de la campaña abierta
(`CAMPANA_2027_1`), cuyo resultado aún no existe y es lo que el modelo predecirá.

In [ ]:
CONTROL = ["inscripcion_id", "periodo", "conjunto"]
OBJETIVO = "matriculado"
NUMERICAS = ["edad_al_inscribirse", "estrato", "distancia_campus_km", "distancia_vial_km", "dias_registro_a_inscripcion",
             "contactos_antes_de_inscribirse", "colegio_promedio_saber11", "anios_desde_saber11",
             "punt_lectura_critica", "punt_matematicas", "punt_sociales", "punt_ciencias", "punt_ingles", "punt_global",
             "puntaje_vocacional",
             "pct_descuento_inicial", "valor_semestre_bruto", "valor_neto_primer_semestre"]
BINARIAS = ["captado_en_evento", "colegio_en_convenio", "entrevista_realizada", "cumple_req_saber11",
            "cumple_req_mi_vocacion", "elegible_foes", "apoyo_foes"]
CATEGORICAS = ["programa", "genero", "nacionalidad", "estado_civil", "regimen_salud", "tipo_ubicacion", "ciudad",
               "departamento", "canal", "asesor_id", "colegio_naturaleza", "colegio_caracter", "colegio_bilingue",
               "tipo_colegio", "colegio_nivel", "beca"]
PREDICTORES = NUMERICAS + BINARIAS + CATEGORICAS
assert sorted(CONTROL + [OBJETIVO] + PREDICTORES) == sorted(df.columns), "Columnas inesperadas en el dataset"

# Tipos: la API devuelve JSON; los enteros con vacíos se guardan como Int64 (no como 1.0 / 0.0)
for c in [OBJETIVO, "puntaje_vocacional"]:
    df[c] = pd.to_numeric(df[c]).astype("Int64")
for c in NUMERICAS + BINARIAS:
    df[c] = pd.to_numeric(df[c])
df["asesor_id"] = df["asesor_id"].astype(str)
df = df.sort_values("inscripcion_id").reset_index(drop=True)

print(f"Forma: {df.shape[0]:,} filas × {df.shape[1]} columnas · {len(NUMERICAS)} numéricas, "
      f"{len(BINARIAS)} binarias, {len(CATEGORICAS)} categóricas, 1 objetivo y {len(CONTROL)} de control")
df.head()

In [ ]:
rol = {**{c: "control" for c in CONTROL}, OBJETIVO: "objetivo", **{c: "numérica" for c in NUMERICAS},
       **{c: "binaria 0/1" for c in BINARIAS}, **{c: "categórica" for c in CATEGORICAS}}
resumen = pd.DataFrame({"rol": pd.Series(rol), "tipo_python": df.dtypes.astype(str),
                        "valores_distintos": df.nunique(), "nulos": df.isna().sum(),
                        "pct_nulos": (df.isna().mean() * 100).round(1)}).loc[df.columns]
display(resumen)

chequeos = {
    "inscripcion_id sin duplicados": df["inscripcion_id"].is_unique,
    "distancia al campus calculada en todas las filas": df["distancia_campus_km"].notna().all(),
    "objetivo vacío solo en la campaña 2027-1": (df[OBJETIVO].isna() == (df["conjunto"] == "CAMPANA_2027_1")).all(),
}
for nombre, ok in chequeos.items():
    print(("✔ " if ok else "✘ ") + nombre)
print("\nFilas por conjunto:", df["conjunto"].value_counts().to_dict())

### 2.1 La entrevista vocacional en la base de datos
El formulario de **entrevista de admisión** de Unicervantes tiene preguntas abiertas. Se guardan las de tres
secciones: **Interés por el programa académico** (5 preguntas), **Historia de vida** (7) e **Interés por la
universidad** (1).

Esquema relacional (capa transaccional, cerrada a la API):

```
inscripcion (1) ──< (0..1) entrevista_admision (1) ──< (13) respuesta_entrevista (n) >── (1) pregunta_entrevista
   inscripcion_id            inscripcion_id PK, FK          inscripcion_id + pregunta_id PK       pregunta_id PK
   puntaje_vocacional        fecha_entrevista               respuesta (texto del aspirante)       seccion, orden, texto
```

| Tabla | Una fila por | Qué guarda |
|---|---|---|
| `pregunta_entrevista` | pregunta (13) | Catálogo: sección, orden y texto de cada pregunta del formulario |
| `entrevista_admision` | inscripción con entrevista | Fecha de la entrevista (después de inscribirse, antes de matricularse) |
| `respuesta_entrevista` | inscripción × pregunta | La respuesta abierta del aspirante |

En la capa analítica, `dataset_matricula` conserva el **puntaje vocacional** que da el entrevistador (1 a 5) y la tabla
plana **`dataset_entrevista`** une las respuestas de cada sección en un solo texto, de solo lectura como
`dataset_matricula`. En el Sprint 2 un modelo NLP lee esos textos.

> No hay respuestas reales de entrevistas en datos abiertos: se simulan con un banco de frases en español (supuestos
> S-48 y S-49). Ninguna respuesta tiene nombres ni datos que identifiquen a una persona.

In [ ]:
SECCIONES = {"texto_interes_programa": "Interés por el programa académico", "texto_historia_vida": "Historia de vida",
             "texto_interes_universidad": "Interés por la universidad"}
entrevista = None
if cliente is not None:
    try:
        entrevista = leer_dataset_matricula(cliente, tabla="dataset_entrevista")   # misma API de solo lectura
        origen_entrevista = "Supabase (API de solo lectura)"
    except Exception:
        pass
if entrevista is None or entrevista.empty:                                    # sin Supabase: la misma copia de arriba
    try:
        entrevista, origen_entrevista = cargar_respaldo("dataset_entrevista"), ORIGEN_RESPALDO
    except Exception:
        entrevista = None
if entrevista is None or entrevista.empty:
    print("dataset_entrevista aún no tiene filas en Supabase: se carga con python src/s05_cargar_supabase.py.")
else:
    con_texto = entrevista["texto_interes_programa"].notna()
    print(f"dataset_entrevista: {len(entrevista):,} inscritos · {int(con_texto.sum()):,} con entrevista · origen: {origen_entrevista}")
    largo = entrevista.loc[con_texto, list(SECCIONES)].apply(lambda s: s.str.split().str.len())
    display(largo.describe().round(1).rename(columns=SECCIONES).loc[["mean", "min", "max"]]
            .rename(index={"mean": "palabras (promedio)", "min": "mínimo", "max": "máximo"}))
    ejemplo = entrevista[con_texto].iloc[0]
    for col, nombre in SECCIONES.items():
        print(f"\n{nombre}:\n  {ejemplo[col]}")

## 3. Análisis exploratorio inicial (EDA)
El EDA se hace sobre el conjunto de **ENTRENAMIENTO** (periodos con resultado). La campaña 2027-1 se describe aparte
y no se mira su objetivo, para no tomar decisiones con datos que el modelo deberá predecir.

### 3.1 Estadística descriptiva

In [ ]:
ent = df[df["conjunto"] == "ENTRENAMIENTO"].copy()
camp = df[df["conjunto"] == "CAMPANA_2027_1"].copy()
print(f"Entrenamiento: {len(ent):,} inscritos · Campaña 2027-1: {len(camp):,} inscritos")

num = ent[NUMERICAS].astype(float)
desc = num.describe().T
desc["asimetria"] = num.skew()
desc["curtosis"] = num.kurt()
desc["coef_variacion"] = desc["std"] / desc["mean"]
desc.round(3)

In [ ]:
filas = []
for c in CATEGORICAS + BINARIAS:
    frec = ent[c].value_counts(normalize=True, dropna=False)
    filas.append({"variable": c, "categorias": ent[c].nunique(dropna=True),
                  "mas_frecuente": frec.index[0], "pct_mas_frecuente": round(100 * frec.iloc[0], 1),
                  "segunda": frec.index[1] if len(frec) > 1 else "-",
                  "pct_segunda": round(100 * frec.iloc[1], 1) if len(frec) > 1 else 0.0})
pd.DataFrame(filas).set_index("variable")

In [ ]:
tasa = ent[OBJETIVO].astype(float).mean()
print(f"Tasa de matrícula en entrenamiento: {tasa:.1%} ({int(ent[OBJETIVO].sum()):,} de {len(ent):,} inscritos)")
balance = ent.groupby(["periodo", "programa"])[OBJETIVO].agg(inscritos="size", matriculados="sum", tasa="mean")
balance["tasa"] = balance["tasa"].astype(float).round(3)
balance

#### La matrícula como ensayo de Bernoulli (distribución binomial)
Cada inscrito se matricula (1) o no (0): es un **ensayo de Bernoulli** con probabilidad *p*. En un grupo de *n*
inscritos, el número de matriculados *x* (y de no matriculados *X = n − x*) sigue una **distribución binomial**.

La base se simuló con la tasa de una **muestra de ejemplo** de Ingeniería en Analítica de Datos, periodos 2024-1 a
2026-1 (supuesto S-15; matriculado = todo estado distinto de desertado):

| x (matriculados) | X (no matriculados) | n | p = x/n | q = X/n | Varianza p·q | P(exactamente x) |
|---|---|---|---|---|---|---|
| 43 | 33 | 76 | 56,6% | 43,4% | 0,246 (≈ 25%) | 9,2% |

Las tasas de los periodos de la muestra (33% a 75%) no difieren más de lo esperable por azar (prueba chi-cuadrado; como
4 de los 10 conteos esperados son menores que 5, se confirmó con una prueba exacta por permutación: p-valor 0,17), por
eso se usa la misma *p* en todos los periodos. Cada inscrito simulado se matricula con su propia probabilidad (según su
beca, canal, distancia…), calibrada para que el promedio sea *p* = 43/76. La tabla compara cada periodo con la binomial esperada: media *n·p*, desviación
√(*n·p·q*) y *z* = (*x* − *n·p*) / √(*n·p·q*). Un |*z*| menor que 2 es lo esperable por azar.

In [ ]:
from scipy.stats import binom

from scipy.stats import chi2_contingency

# Muestra de ejemplo (supuesto S-15): conteos agregados por periodo, sin datos personales. [n aspirantes, x matriculados]
MUESTRA = {"2024-1": (22, 16), "2024-2": (9, 4), "2025-1": (8, 6), "2025-2": (9, 3), "2026-1": (28, 14)}
muestra = pd.DataFrame(MUESTRA, index=["n", "x matriculados"]).T
muestra["X no matriculados"] = muestra["n"] - muestra["x matriculados"]
muestra["p = x/n"] = (muestra["x matriculados"] / muestra["n"]).round(3)
N_MUESTRA, X_MUESTRA = int(muestra["n"].sum()), int(muestra["x matriculados"].sum())   # 76 y 43
P = X_MUESTRA / N_MUESTRA
Q = 1 - P
muestra.loc["Total"] = [N_MUESTRA, X_MUESTRA, N_MUESTRA - X_MUESTRA, round(P, 3)]
display(muestra)
chi2, p_valor, gl, _ = chi2_contingency(muestra.iloc[:-1][["x matriculados", "X no matriculados"]].to_numpy())
print(f"Muestra de ejemplo: x = {X_MUESTRA}, X = {N_MUESTRA - X_MUESTRA}, n = {N_MUESTRA} -> p = {P:.1%}, q = {Q:.1%}, "
      f"varianza p·q = {P * Q:.3f}, P(exactamente {X_MUESTRA}) = {binom.pmf(X_MUESTRA, N_MUESTRA, P):.1%}")
print(f"¿La tasa cambia entre periodos? chi-cuadrado = {chi2:.2f} (gl = {gl}), p-valor = {p_valor:.2f}: "
      + ("no hay evidencia de diferencias, se usa una sola p." if p_valor >= 0.05 else "sí hay diferencias entre periodos."))

def resumen_binomial(g: pd.DataFrame) -> dict:
    n, x = len(g), int(g[OBJETIVO].sum())
    return {"n": n, "x matriculados": x, "X no matriculados": n - x, "p = x/n": x / n, "q = X/n": 1 - x / n,
            "varianza p·q": x / n * (1 - x / n), "esperado n·p": n * P, "desviación √(n·p·q)": (n * P * Q) ** 0.5,
            "z": (x - n * P) / (n * P * Q) ** 0.5}

historico = df[df[OBJETIVO].notna()]
tabla_binomial = pd.DataFrame({per: resumen_binomial(g) for per, g in historico.groupby("periodo")}).T
tabla_binomial.loc["Todos (2024-1 a 2026-2)"] = resumen_binomial(historico)
display(tabla_binomial.astype(float).round({"n": 0, "x matriculados": 0, "X no matriculados": 0, "p = x/n": 3, "q = X/n": 3,
                                            "varianza p·q": 3, "esperado n·p": 1, "desviación √(n·p·q)": 1, "z": 2}))

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))
k = np.arange(25, 62)
ejes[0].bar(k, binom.pmf(k, N_MUESTRA, P), color=[NARANJA if v == X_MUESTRA else AZUL for v in k])
ejes[0].set_title(f"Binomial(n = {N_MUESTRA}, p = {P:.3f}): P(x = {X_MUESTRA}) = {binom.pmf(X_MUESTRA, N_MUESTRA, P):.1%}")
ejes[0].set_xlabel("x = número de matriculados")
ejes[0].set_ylabel("probabilidad")
periodos = tabla_binomial.index[:-1]
n_per = tabla_binomial.loc[periodos, "n"].astype(float)
bajo, alto = binom.ppf(0.025, n_per, P) / n_per, binom.ppf(0.975, n_per, P) / n_per
ejes[1].fill_between(range(len(periodos)), bajo, alto, color=GRIS, alpha=0.3, label="95% esperado por la binomial")
ejes[1].plot(range(len(periodos)), tabla_binomial.loc[periodos, "p = x/n"].astype(float), "o-", color=AZUL, label="tasa simulada")
ejes[1].axhline(P, color=NARANJA, ls="--", label=f"p = 43/76 = {P:.1%}")
ejes[1].set_xticks(range(len(periodos)), periodos)
ejes[1].set_ylim(0.4, 0.75)
ejes[1].set_title("Tasa de matrícula por periodo frente a la binomial")
ejes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lectura de la estadística descriptiva**
- El objetivo está **razonablemente balanceado**: cerca del 57% de los inscritos se matricula (la *p* = 43/76 de la
  muestra de ejemplo) y cada periodo queda dentro de lo que permite la binomial. No hace falta sobremuestreo, pero
  conviene validar por periodo.
- Los puntajes Saber 11 tienen asimetría cercana a 0: son casi simétricos.
- La **distancia al campus** (asimetría ≈ 0,9), los **días entre registro e inscripción** (≈ 2,5), la **edad** (≈ 1,8) y
  los **años desde el Saber 11** (≈ 2,4) tienen cola larga a la derecha. Son candidatas a transformación logarítmica
  o a agruparse en rangos en el Sprint 2.
- Variables como `valor_semestre_bruto` casi no varían dentro de un programa y un año: aportan poco por sí solas,
  pero el **valor neto** (después de la beca) sí varía.

### 3.2 Distribuciones

In [ ]:
variables = ["punt_global", "distancia_vial_km", "distancia_campus_km", "edad_al_inscribirse",
             "valor_neto_primer_semestre", "dias_registro_a_inscripcion"]
fig, ejes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, c in zip(ejes.flat, variables):
    x = ent[c].astype(float)
    ax.hist(x, bins=35, color=AZUL, alpha=0.85)
    ax.axvline(x.mean(), color=NARANJA, lw=2, label=f"media {x.mean():,.1f}")
    ax.axvline(x.median(), color=GRIS, lw=2, ls="--", label=f"mediana {x.median():,.1f}")
    ax.set_title(c)
    ax.legend(fontsize=8)
fig.suptitle("Distribución de las principales variables numéricas (entrenamiento)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
ejes[0].hist(np.log(ent["distancia_campus_km"].astype(float)), bins=30, color=AZUL, alpha=0.85)
ejes[0].set_title("log(distancia al campus)")
ejes[1].boxplot([ent.loc[ent["tipo_colegio"] == t, "punt_global"] for t in ["OFICIAL", "PRIVADO", "TECNICO"]])
ejes[1].set_xticks([1, 2, 3], ["Público", "Privado", "Técnico"])
ejes[1].set_title("Saber 11 global por tipo de colegio")
ejes[2].boxplot([ent.loc[ent["beca"] == b, "valor_neto_primer_semestre"] / 1e6 for b in ["NINGUNA", "MI_VOCACION", "SABER_11"]])
ejes[2].set_xticks([1, 2, 3], ["NINGUNA", "MI_VOCACION", "SABER_11"])
ejes[2].set_title("Valor neto del 1er semestre (millones COP) por beca")
plt.tight_layout()
plt.show()

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, c in zip(ejes.flat, ["canal", "beca", "estrato", "tipo_colegio", "programa", "genero"]):
    conteo = ent[c].value_counts().sort_values()
    if c == "tipo_colegio":                     # el ICFES dice OFICIAL; para leer se muestra "PÚBLICO"
        conteo = conteo.rename({"OFICIAL": "PÚBLICO", "TECNICO": "TÉCNICO"})
    conteo.plot(kind="barh", ax=ax, color=AZUL)
    ax.set_title(c)
fig.suptitle("Frecuencias de las principales variables categóricas (entrenamiento)")
plt.tight_layout()
plt.show()

In [ ]:
def atipicos_iqr(s: pd.Series) -> dict:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    bajo, alto = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {"limite_inferior": bajo, "limite_superior": alto, "por_debajo": int((s < bajo).sum()),
            "por_encima": int((s > alto).sum()), "pct_atipicos": round(100 * ((s < bajo) | (s > alto)).mean(), 1)}

atip = pd.DataFrame({c: atipicos_iqr(ent[c].astype(float).dropna()) for c in NUMERICAS}).T
atip[atip["pct_atipicos"] > 0].sort_values("pct_atipicos", ascending=False)

**Lectura de las distribuciones**
- El 42% de los inscritos vive a 8 km o menos del campus en línea recta (sobre todo en Suba, Usaquén, Chía y Cota). Por
  vías, el 36% está a 10 km o menos y el 28% a más de 20 km: una cola larga hacia el centro y el sur de Bogotá (hasta
  52 km por vía) y el norte de la Sabana. En escala logarítmica la distribución es más simétrica.
- **Distancia por dirección:** cada inscrito tiene una dirección real (placa de Catastro Bogotá o dirección del DANE en
  Sabana Centro, asignada al azar a una persona ficticia). La distancia se calcula desde esa coordenada, en línea recta
  y por vías (estimada: tramo norte-sur + oriente-occidente, S-44); por eso hay casi un valor distinto por inscrito.
- Los colegios privados tienen puntajes Saber 11 más altos que los públicos. Es el mismo patrón de los datos reales
  del ICFES.
- El valor neto se separa en grupos por beca: la beca baja el valor del primer semestre en 30%.
- Los **atípicos** son valores posibles: personas adultas, residencias lejanas, puntajes muy altos.
  **No se eliminan**

### 3.3 Valores nulos
No todos los vacíos significan lo mismo. Se clasifican antes de tratarlos:

In [ ]:
TIPO_NULO = {
    "matriculado": ("Censura", "Campaña 2027-1: el resultado aún no ocurre", "No se imputa: es el conjunto a predecir"),
    "puntaje_vocacional": ("Estructural", "No hubo entrevista con psicología", "0 = sin valoración (entrevista_realizada ya lo indica)"),
    "colegio_caracter": ("Faltante real (ICFES)", "El ICFES no reporta el carácter de ese colegio", "Categoría NO_REPORTA"),
    "colegio_bilingue": ("Faltante real (ICFES)", "El ICFES no reporta si el colegio es bilingüe", "Categoría NO_REPORTA"),
}
nulos = df.isna().sum()
nulos = nulos[nulos > 0]
tabla_nulos = pd.DataFrame({"nulos": nulos, "pct": (100 * nulos / len(df)).round(1),
                            "nulos_entrenamiento": ent[nulos.index].isna().sum(),
                            "nulos_campana": camp[nulos.index].isna().sum()})
tabla_nulos[["tipo", "por_que", "tratamiento"]] = [TIPO_NULO.get(c, ("Sin clasificar", "-", "-")) for c in tabla_nulos.index]
display(tabla_nulos)

fig, ax = plt.subplots(figsize=(9, 3))
tabla_nulos["pct"].sort_values().plot(kind="barh", ax=ax, color=NARANJA)
ax.set_xlabel("% de filas vacías (dataset completo)")
ax.set_title("Columnas con valores nulos")
plt.tight_layout()
plt.show()

In [ ]:
# ¿Los vacíos son como se dijo? Se comprueba en vez de suponerlo
print("puntaje_vocacional vacío ⇔ sin entrevista:",
      (df["puntaje_vocacional"].isna() == (df["entrevista_realizada"] == 0)).all())
print("matriculado vacío ⇔ campaña 2027-1:", (df[OBJETIVO].isna() == (df["conjunto"] == "CAMPANA_2027_1")).all())
print("\nColegios sin carácter o sin dato de bilingüismo, por naturaleza (% de inscritos):")
print(df.assign(sin_caracter=df["colegio_caracter"].isna(), sin_bilingue=df["colegio_bilingue"].isna())
        .groupby("colegio_naturaleza")[["sin_caracter", "sin_bilingue"]].mean().mul(100).round(1))

### 3.4 Tratamiento preliminar de nulos
Decisiones (todas reversibles; el dataset de Supabase no se modifica):
1. **Objetivo vacío:** no se imputa. La campaña 2027-1 se separa como conjunto de predicción.
2. **`puntaje_vocacional` vacío:** se reemplaza por **0 = sin valoración**. No se usa la media porque el vacío tiene
   significado (no hubo entrevista) y `entrevista_realizada` ya lo marca.
3. **`colegio_caracter` y `colegio_bilingue` vacíos:** nueva categoría **`NO_REPORTA`**. Imputar la moda inventaría
   un dato que el ICFES no tiene.

En el Sprint 3 este tratamiento quedó dentro del *pipeline* de scikit-learn (`modelado.preparar_X` y
`modelado.construir_pipeline`). Así se aplica igual en entrenamiento, validación y campaña, sin fuga de información.

In [ ]:
def tratamiento_preliminar(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    d["puntaje_vocacional"] = d["puntaje_vocacional"].fillna(0).astype(int)
    for c in ["colegio_caracter", "colegio_bilingue"]:
        d[c] = d[c].fillna("NO_REPORTA")
    return d

ent_t, camp_t = tratamiento_preliminar(ent), tratamiento_preliminar(camp)
antes = ent[PREDICTORES].isna().sum()
despues = ent_t[PREDICTORES].isna().sum()
print(pd.DataFrame({"antes": antes[antes > 0], "despues": despues[antes > 0]}))
assert despues.sum() == 0 and camp_t[PREDICTORES].isna().sum().sum() == 0
print(f"\nPredictores sin nulos: entrenamiento {ent_t.shape} y campaña {camp_t.shape}. "
      "El objetivo de la campaña sigue vacío a propósito.")

### 3.5 Primera mirada a la relación con la matrícula
Es un primer acercamiento (correlaciones y tasas por grupo), no el modelo. Sirve para priorizar variables en el
Sprint 2.

In [ ]:
t = ent_t.assign(quintil_distancia=pd.qcut(ent_t["distancia_vial_km"].rank(method="first"), 5,
                                           labels=["Q1 cerca", "Q2", "Q3", "Q4", "Q5 lejos"]),
                quintil_saber11=pd.qcut(ent_t["punt_global"].rank(method="first"), 5,
                                        labels=["Q1 bajo", "Q2", "Q3", "Q4", "Q5 alto"]))
t[OBJETIVO] = t[OBJETIVO].astype(float)
fig, ejes = plt.subplots(2, 2, figsize=(14, 8))
for ax, c in zip(ejes.flat, ["beca", "canal", "quintil_distancia", "quintil_saber11"]):
    tasas = t.groupby(c, observed=True)[OBJETIVO].mean()
    if c == "canal":
        tasas = tasas.sort_values()
    tasas.plot(kind="barh" if c == "canal" else "bar", ax=ax, color=AZUL)
    (ax.axvline if c == "canal" else ax.axhline)(t[OBJETIVO].mean(), color=NARANJA, ls="--", label="tasa global")
    ax.set_title(f"Tasa de matrícula por {c}")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
corr = (t[NUMERICAS + BINARIAS].astype(float).corrwith(t[OBJETIVO]).dropna()
        .sort_values(key=abs, ascending=False))
fig, ax = plt.subplots(figsize=(9, 7))
corr.sort_values().plot(kind="barh", ax=ax, color=[NARANJA if v < 0 else AZUL for v in corr.sort_values()])
ax.set_title("Correlación de cada variable numérica o binaria con la matrícula")
ax.set_xlabel("Correlación de Pearson con matriculado (punto-biserial)")
plt.tight_layout()
plt.show()
corr.round(3).to_frame("correlacion_con_matricula").head(10)

**Lectura**
- **Beca y valor neto:** con beca se matricula entre el 67% y el 69%; sin beca, cerca del 43%. El descuento baja
  el precio del primer semestre.
- **Canal:** inmersión (≈ 70%), referidos, Campus Day y colegio en convenio superan a ExpoEstudiantes, llamadas y
  redes sociales (≈ 40-48%).
- **Distancia (por vías, desde la dirección):** la tasa baja del 65% en el quintil más cercano al 48% en el más lejano.
- **Saber 11:** la tasa sube con el puntaje (44% a 66% por quintil) porque los puntajes altos reciben la Beca Saber 11;
  el modelo separa ese efecto de la "U invertida" (Sprint 4).
- Las correlaciones individuales son **bajas** (la mayor, la del valor neto y el descuento, ronda 0,26), como se espera en un problema
  de comportamiento: ninguna variable sola decide la matrícula.

> Recordatorio: el resultado de matrícula es sintético. Estos patrones reflejan los supuestos documentados del
> generador; con los datos reales de Admisiones el mismo cuaderno se vuelve a ejecutar sin cambios.

## 4. Conclusiones del Sprint 1 y próximos pasos
**Lo que se logró**
1. **Conexión segura:** Python lee Supabase con la API. Las credenciales están en `.env` o en los Secretos de Colab, nunca
   en el código. La clave publicable **solo puede leer** el dataset y la base rechaza cualquier escritura.
2. **Dataset plano:** `dataset_matricula` tiene una fila por inscrito, 45 columnas (3 de control, 1 objetivo y 41
   predictores), la distancia al campus ya calculada, y no tiene datos personales, notas del colegio ni variables con
   fuga.
3. **EDA inicial:** estadística descriptiva, distribuciones, atípicos marcados y nulos clasificados y tratados. Los
   predictores de entrenamiento y de campaña quedan sin vacíos.

**FIN DEL SPRINT 1**

---
**COMIENZA EL SPRINT 2** · entrega del jueves 29 de octubre de 2026 · archivos del sprint en `sprints/sprint_2_caracteristicas/`

# Sprint 2 — Ingeniería de características y regresión logística
**Proyecto:** Modelo Predictivo de Segmentación de Mercado y Propensión a la Matrícula · Unicervantes

**Pedido del docente (Semana 2):** entregar *la misma tabla de datos con columnas nuevas y útiles* (por ejemplo,
"¿Aplica a Beca Mi Vocación? Sí/No" o `segmento_rendimiento_icfes`) y *un primer modelo que sirva de punto de
comparación*. El proyecto predice **solo la matrícula**; el modelo base es una **regresión logística**.

| Sección | Contenido |
|---|---|
| 1 | Datos y partición temporal (entrenamiento, validación, prueba y campaña) |
| 2 | La entrevista de admisión en texto y el modelo NLP del puntaje vocacional |
| 3 | Lo que se aprende SOLO del entrenamiento (sin fuga de datos) |
| 4 | Las 14 columnas nuevas |
| 5 | Segmentación de mercado con k-means |
| 6 | Relación de las columnas nuevas con la matrícula |
| 7 | Regresión logística y su error |

## 1. Datos y partición temporal
Los datos se dividen por **tiempo**, no al azar: se aprende con el pasado y se evalúa con el futuro, como pasará en la
realidad. La campaña 2027-1 no tiene resultado (se predice); su resultado real está sellado hasta el Sprint 5.

| Conjunto | Periodos | Para qué |
|---|---|---|
| ENTRENAMIENTO | 2024-1 a 2025-2 | Aprender escalas, k-means y modelos |
| VALIDACION | 2026-1 | Elegir hiperparámetros (Sprint 3) |
| PRUEBA | 2026-2 | Medir el error final, una sola vez |
| CAMPANA | 2027-1 | Lo que el modelo predice |

In [ ]:
import caracteristicas as ca
import flujo
import modelado as mo

# Lee las tablas planas por la API de Supabase (solo lectura): dataset_matricula y las respuestas de la entrevista
# (dataset_entrevista). Sin clave de Supabase usa la copia de esas tablas: en el computador, la del repositorio; en
# Colab, la que está en GitHub (no hay que subir nada). Después construye las columnas del Sprint 2 (con el puntaje NLP).
f = flujo.cargar_y_construir("auto")
d, ref, cfg = f["inscritos"], f["referencias"], ca.configuracion()
print("Origen de los datos:", f["origen"])
print(f"Inscritos: {len(d):,} · con entrevista: {int(d['entrevista_realizada'].sum()):,}")
print(d["particion"].value_counts().to_string())

## 2. La entrevista de admisión en texto y el modelo NLP
La entrevista de admisión es **cualitativa**: el aspirante responde con sus palabras. Se usan tres secciones del
formulario de Unicervantes, con 13 preguntas abiertas:

| Sección | Preguntas |
|---|---|
| Interés por el programa académico | Por qué la carrera, defina el programa, qué le gusta, cómo se ve como profesional, otra carrera |
| Historia de vida | Sobre usted, ocupación del hogar, trabajo y financiación, cualidades, aspectos por mejorar, colegio, tiempo libre |
| Interés por la universidad | ¿Por qué quiere estudiar en Unicervantes? |

El entrevistador da un **puntaje vocacional de 1 a 5** (está en `dataset_matricula`). Un **modelo NLP** lee las
respuestas y aprende qué palabras acompañan a cada puntaje:

1. **Preprocesamiento:** minúsculas, sin tildes, sin palabras vacías ("de", "la", "que"…), pero conservando las que
   cambian el sentido ("no", "todavía", "ya"). Se forman palabras y pares de palabras dentro de cada frase.
2. **TF-IDF:** cada término pesa más si aparece mucho en esa entrevista y poco en las demás. Un TF-IDF por sección.
3. **Regresión Ridge:** aprende el peso de cada término para estimar el puntaje del entrevistador. Ridge castiga los
   pesos grandes (α elegido con validación cruzada **dentro** del entrenamiento).

El resultado, `puntaje_vocacional_nlp` (1 a 5), es la vocación según las palabras del propio aspirante. No copia al
entrevistador: las palabras también muestran lo que él no alcanza a ver, y por eso pueden ayudar a predecir la
matrícula (Sprint 3).

- **Sin fuga de datos:** el NLP se ajusta solo con el entrenamiento. Las entrevistas del entrenamiento reciben una
  predicción **fuera de pliegue** (validación cruzada de 5 pliegues): ninguna se puntúa con un modelo que ya la vio.

> No hay respuestas reales de entrevistas en datos abiertos: se simulan con un banco de frases en español (supuestos
> S-48 y S-49, `src/textos_entrevista.py`).

In [ ]:
import caracteristicas as ca
import nlp_vocacion as nv

print(f"Inscritos con entrevista: {int(d['entrevista_realizada'].sum()):,} de {len(d):,} (13 respuestas cada uno)")
preguntas = pd.DataFrame(cfg["generador"]["entrevista_texto"]["preguntas"])
display(preguntas.rename(columns={"id": "pregunta", "seccion": "sección", "texto": "pregunta del formulario"}))
textos = f["entrevista"]                     # las respuestas por sección: la entrada del NLP
ejemplo = textos[textos["texto_interes_programa"].notna()].iloc[0]
for col, nombre in nv.SECCIONES.items():
    print(f"\n{nombre}:\n  {ejemplo[col]}")
print("\nCómo lo ve el modelo (primera sección, primeros 20 términos):", nv.terminos(ejemplo["texto_interes_programa"])[:20])

### Qué aprendió el modelo NLP
Las palabras con peso positivo suben el puntaje vocacional; las de peso negativo lo bajan. Se mide en la validación
(2026-1), que el NLP nunca vio: error absoluto medio (MAE) frente al puntaje del entrevistador, comparado con predecir
siempre el promedio, y la correlación de Spearman (orden).

In [ ]:
import graficas_sprint2 as gr
from sklearn.metrics import roc_auc_score


def ver(fig):
    """Muestra la figura en su lugar (antes de las tablas que siguen) y la cierra."""
    display(fig)
    plt.close(fig)


nlp = ref["nlp"]
print(f"Entrevistas de entrenamiento: {nlp['n_entrevistas']:,} · términos: {nlp['n_terminos']:,} · α de Ridge: {nlp['alpha']}")
ver(gr.palabras_nlp(nv.palabras_clave(nlp)))
filas = {}
for conj in ["ENTRENAMIENTO", "VALIDACION"]:
    x = flujo.conjunto(d, conj).dropna(subset=["puntaje_vocacional_nlp"])
    filas[conj] = nv.metricas(x["puntaje_vocacional"], x["puntaje_vocacional_nlp"], nlp["media_entrenamiento"]) | {
        "AUC del NLP solo (matrícula)": roc_auc_score(x["matriculado"].astype(int), x["puntaje_vocacional_nlp"])}
display(pd.DataFrame(filas).T.round(3))
ver(gr.puntaje_nlp(d, "VALIDACION"))
v = filas["VALIDACION"]
print(f"En la validación el NLP se equivoca {v['mae']:.2f} puntos en promedio frente al entrevistador "
      f"(predecir siempre el promedio: {v['mae_promedio']:.2f}).")

## 3. Lo que se aprende solo del entrenamiento
Si las escalas o los grupos se calcularan con todos los datos, la validación y la prueba "verían" información del
futuro (fuga de datos) y el error saldría engañosamente bajo. Por eso `ajustar_referencias` usa **solo** el
entrenamiento: la media y la desviación del Saber 11 (para su término cuadrático), los grupos de k-means y el modelo NLP.

In [ ]:
print("Estandarización (media y desviación del entrenamiento):")
display(pd.DataFrame(ref["estandarizacion"], index=["media", "desviación"]).round(3))

## 4. Las columnas nuevas
Cada columna responde una pregunta de negocio. Las que dependen de una regla (Sí/No de becas, bandas del ICFES,
valor neto) se calculan igual para todos; las que dependen de datos (k-means y el puntaje NLP) usan lo aprendido en
el entrenamiento.

In [ ]:
print(f"Columnas nuevas: {len(ca.NUEVAS)}")
display(pd.DataFrame({"columna": list(ca.DICCIONARIO), "qué es": list(ca.DICCIONARIO.values())}))
muestra = d[["inscripcion_id", "periodo", "punt_global", "aplica_beca_saber11", "aplica_beca_mi_vocacion",
             "puntaje_vocacional", "puntaje_vocacional_nlp", "segmento_rendimiento_icfes",
             "valor_neto_primer_semestre", "ahorro_beca_primer_semestre", "zona_distancia", "segmento_mercado"]]
display(muestra.head(8))
print("¿Aplica a Mi Vocación?", d["aplica_beca_mi_vocacion"].value_counts().to_dict(),
      "· ¿Aplica a Saber 11?", d["aplica_beca_saber11"].value_counts().to_dict())
print("Nulos en las columnas nuevas:", int(d[ca.NUEVAS].drop(columns="puntaje_vocacional_nlp").isna().sum().sum()),
      "· puntaje NLP vacío solo sin entrevista:", int(d["puntaje_vocacional_nlp"].isna().sum()))

## 5. Segmentación de mercado con k-means
k-means agrupa a los aspirantes por parecido en su perfil (estrato, distancia, ICFES, edad y tipo de colegio). No se
usan la media ICFES del colegio (Admisiones no suele tener ese dato) ni las notas del colegio (dejaron de ser variables
del análisis). Antes se **estandarizan** las variables (restar la media y dividir por la desviación), para que ninguna pese
más solo por estar en otra escala. El número de grupos k se elige con la **silueta**: qué tan cerca está cada aspirante
de su grupo frente al grupo vecino (de −1 a 1; más alto es mejor).

In [ ]:
ver(gr.eleccion_k(ref))
print("Mapa de los segmentos: las 6 variables proyectadas en 2 dimensiones (componentes principales).")
ver(gr.mapa_segmentos(d, ref))
print("Cada segmento se nombra por los 2 rasgos en que más se aleja del promedio de los inscritos.")
ver(gr.perfil_segmentos(d))
ver(gr.tamano_y_matricula(d))
print("Tabla de segmentos (promedios reales de cada grupo):")
display(ca.tabla_segmentos(d))

**Cómo leer el mapa.** k-means usa 5 variables a la vez, así que no se puede dibujar directamente. El mapa las resume en
2 dimensiones (componentes principales): cada punto es un inscrito, el color es el segmento que le asignó k-means y el
círculo grande es el centro del grupo. Los grupos se tocan en los bordes: por eso la silueta es baja.

**Cómo leer los nombres.** Cada segmento se nombra con los **2 rasgos en que más se diferencia del promedio**
de todos los inscritos. Los nombres son **relativos**: "viven cerca del campus" quiere decir más cerca que el promedio
(15,6 km por vías), no a menos de cierta distancia; "colegio público" quiere decir que casi todos vienen de colegios
oficiales (públicos). La tabla muestra los valores reales para comprobarlo.

| Segmento | En palabras sencillas |
|---|---|
| S1: Colegio privado, estrato más alto | El grupo más grande: colegios privados, estratos más altos y el mejor ICFES (muchos aplican a la beca Saber 11); el que más se matricula |
| S2: Colegio público, viven cerca del campus | Egresados de colegios públicos que viven relativamente cerca (unos 10 km por vías) |
| S3: Viven lejos del campus, estrato más bajo | Viven a unos 29 km por vías y son de estratos 1 y 2; el que menos se matricula |
| S4: Aspirantes mayores, puntaje ICFES bajo | Unos 22 años en promedio: no vienen directo del colegio |

**Qué tan buenos son los grupos.** La silueta es baja (cerca de 0,24): los aspirantes no forman grupos muy separados,
algo común en datos de personas. Aun así, los segmentos tienen perfiles con sentido de negocio y tasas de matrícula
distintas (del 49% al 61%), por eso sirven para orientar la estrategia de captación (Sprint 5).

## 6. Relación de las columnas nuevas con la matrícula (entrenamiento)

In [ ]:
ent = flujo.conjunto(d, "ENTRENAMIENTO")
columnas = ["segmento_rendimiento_icfes", "aplica_beca_mi_vocacion", "canal_agrupado", "zona_distancia", "estrato_grupo",
            "rezago_saber11"]
ver(gr.matricula_por_grupo(ent, columnas))
display(gr.tabla_matricula_por_grupo(ent, columnas))
correl = ent[["puntaje_vocacional", "puntaje_vocacional_nlp", "valor_neto_primer_semestre", "log_distancia"]].corrwith(
    ent["matriculado"].astype(float))
display(correl.rename(ca.DICCIONARIO).round(3).rename("correlación con la matrícula").to_frame())

**Lectura.**
- El segmento ICFES ALTO (310-349) se matricula más que el SUPERIOR (≥ 350): los puntajes muy altos tienen más
  opciones puesto a que las universidades los buscan. Es la "U invertida" que el modelo deberá captar.
- Quien aplica a Mi Vocación se matricula más: la beca baja el precio.
- Los canales relacionales (referido, inmersión, convenio) convierten más que los eventos y lo digital.
- La matrícula baja con la distancia al campus (por vías, desde la dirección de cada inscrito).
- Quien presentó la entrevista se matricula más que quien no la presentó. Entre los entrevistados, el puntaje NLP se
  relaciona con la matrícula más que el puntaje del entrevistador: las palabras muestran algo más que su calificación.

## 7. Regresión logística: el primer modelo
La regresión logística estima la **probabilidad** de que un inscrito se matricule a partir de todas sus variables:

`log(p / (1 − p)) = β₀ + β₁x₁ + … + βₖxₖ`

Cada β dice cuánto sube o baja la probabilidad cuando la variable cambia. Las variables numéricas se estandarizan y las
categóricas se codifican con *one-hot* (una columna 0/1 por categoría), dentro del mismo modelo.

- Se entrena solo con el **entrenamiento** (2024-1 a 2025-2) y **sin ajustar hiperparámetros**: C = 1, fijado antes de
  ver resultados (configuración del proyecto, `modelado.sprint2_logistica`). El ajuste se hace en el Sprint 3.
- Se predice "se matricula" si la probabilidad es ≥ 50%. El **error %** es el porcentaje de inscritos mal clasificados.
- El **AUC** mide qué tan bien ordena a los inscritos: 0,5 es azar y 1 es perfecto.
- La entrevista entra con el **puntaje del entrevistador** (0 si no hubo entrevista). El puntaje NLP se agrega en el
  Sprint 3, para medir cuánto mejora el modelo.
- Es el **modelo base** del proyecto: el Sprint 3 lo compara con modelos más avanzados.

In [ ]:
V = mo.VARIABLES_MATRICULA
ent = flujo.conjunto(d, "ENTRENAMIENTO")
logistica = mo.construir_pipeline("logistica", cfg["modelado"]["sprint2_logistica"], V, cfg["modelado"]["semilla"]).fit(
    mo.preparar_X(ent, ref, V), ent["matriculado"].astype(int))
coef = pd.Series(logistica.named_steps["modelo"].coef_[0], index=logistica.named_steps["preparacion"].get_feature_names_out())
print("Los 10 coeficientes más grandes (variables estandarizadas; positivo = sube la probabilidad):")
display(coef.reindex(coef.abs().sort_values(ascending=False).index).head(10).round(3).rename("β").to_frame())
resultados = {}
for conj in ["VALIDACION", "PRUEBA"]:
    X = flujo.conjunto(d, conj)
    resultados[conj] = mo.metricas(X["matriculado"], logistica.predict_proba(mo.preparar_X(X, ref, V))[:, 1])
metr = pd.DataFrame(resultados).T.astype(float)
display(metr[["n", "error_pct", "auc_roc", "brier"]].round(3))
ver(gr.error_logistica(metr))
b = resultados["PRUEBA"]
print(f"\nLa regresión logística se equivoca en el {b['error_pct']:.1f}% de los inscritos de 2026-2 (AUC {b['auc_roc']:.3f}). "
      "Este es el punto de comparación del Sprint 3.")

**FIN DEL SPRINT 2**

---
**COMIENZA EL SPRINT 3** · entrega del jueves 5 de noviembre de 2026 · archivos del sprint en `sprints/sprint_3_modelado/`

# Sprint 3 — Modelado: la regresión logística mejorada con el NLP de la entrevista, frente a Random Forest y XGBoost
**Pedido del docente (Semana 3):** *un modelo de Machine Learning que ya haga predicciones* y una tabla comparativa
sencilla: *"El cálculo de la Semana 2 tenía un error del X%, mientras que este modelo tiene un error del Y%"*.

El proyecto predice **si el inscrito se matricula**. El Sprint 2 dejó una regresión logística sin ajustar y el
**puntaje vocacional NLP** (lo que dicen las palabras del aspirante en la entrevista). Aquí la logística se mejora con
ese puntaje y con el ajuste de sus hiperparámetros, y se compara con dos modelos de árboles, Random Forest y XGBoost.

| Sección | Contenido |
|---|---|
| 1 | Variables de entrada (sin fuga de datos) |
| 2 | Ajuste de hiperparámetros con validación temporal |
| 3 | Comparación final en la prueba 2026-2 |
| 4 | Modelos finales y tabla de predicciones (bloque 3 del tablero) |

## 1. Variables de entrada
Solo se usan datos que existen **antes** de la decisión que se predice. Por ejemplo, los documentos entregados o la
forma de financiación no entran: se conocen al matricularse y serían fuga de datos.
Se agregan dos variables simples: `icfes_al_cuadrado`, para que la regresión logística pueda captar la "U invertida"
del Saber 11, y `semestre_2`, el ingreso a mitad de año.

La entrevista entra de dos formas: `puntaje_vocacional` (el del entrevistador, como en el Sprint 2) y
`puntaje_vocacional_nlp` (el que el modelo NLP del Sprint 2 lee en las palabras del aspirante). En ambas, 0 = sin
entrevista, y `entrevista_realizada` distingue ese caso.

In [ ]:
V, V2 = mo.VARIABLES_MATRICULA_NLP, mo.VARIABLES_MATRICULA      # Sprint 3 (con NLP) · Sprint 2 (sin NLP)
display(pd.DataFrame([(tipo, ", ".join(cols)) for tipo, cols in V.items()], columns=["tipo", "variables (matrícula)"]))
y = lambda x: x["matriculado"].astype(int)
X = lambda x: mo.preparar_X(x, ref, V)
X2 = lambda x: mo.preparar_X(x, ref, V2)
e, v, t = (flujo.conjunto(d, c) for c in ["ENTRENAMIENTO", "VALIDACION", "PRUEBA"])
ev = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION")
print(f"Entrenamiento {len(e)} · validación {len(v)} · prueba {len(t)} inscritos")

## 2. Ajuste de hiperparámetros con validación temporal
Cada modelo tiene "perillas" (hiperparámetros): en la regresión logística, la regularización C y el tipo de
penalización (L2 achica todos los pesos; L1 puede dejar en cero las variables que no aportan); la profundidad de los
árboles en Random Forest; la tasa de aprendizaje en XGBoost. Cada combinación se **entrena con 2024-1 a 2025-2 y se
mide en 2026-1**. Se elige la de menor **log-loss**, que castiga más las probabilidades seguras que salen mal.
Si un modelo más simple queda a menos de 0,002 del mejor, se prefiere el simple (navaja de Occam).

**Control:** la misma búsqueda de la logística, pero **sin** el puntaje NLP. Así se mide cuánto aporta el texto de la
entrevista, separado del ajuste de hiperparámetros.

In [ ]:
COMPLEJIDAD = ["logistica", "random_forest", "xgboost"]
busquedas = {tipo: mo.ajustar_temporal(tipo, cfg, X(e), y(e), X(v), y(v), V) for tipo in COMPLEJIDAD}
for tipo, b in busquedas.items():
    print(f"\n{tipo}: {len(b)} combinaciones probadas; las 3 mejores en validación 2026-1")
    display(b.drop(columns="parametros").head(3).round(4))
mejores = {tipo: b.iloc[0] for tipo, b in busquedas.items()}
sin_nlp = mo.ajustar_temporal("logistica", cfg, X2(e), y(e), X2(v), y(v), V2)
print(f"\nControl sin NLP: mejor log-loss {sin_nlp.iloc[0]['log_loss_validacion']:.4f} {sin_nlp.iloc[0]['parametros']} · "
      f"con NLP: {mejores['logistica']['log_loss_validacion']:.4f} {mejores['logistica']['parametros']}")
minimo = min(m["log_loss_validacion"] for m in mejores.values())
elegido = next(tp for tp in COMPLEJIDAD if mejores[tp]["log_loss_validacion"] <= minimo + 0.002)
print(f"\nModelo elegido por validación: {elegido} {mejores[elegido]['parametros']}")

**Lectura.**
- Con el puntaje NLP, la log-loss de la logística en la validación baja claramente frente al control sin NLP: las
  palabras del aspirante aportan información que el puntaje del entrevistador no tiene.
- La regresión logística gana a los árboles: la relación entre las variables y la matrícula es casi lineal en log-odds,
  y en ese caso los árboles no tienen patrones complejos que aprovechar y corren más riesgo de sobreajustarse.
- Ojo: la matrícula de estos datos se simuló con un modelo logístico (supuesto S-15) y el texto con el supuesto S-49;
  con entrevistas y matrículas reales la comparación se repite y puede cambiar.

## 3. Comparación final en la prueba 2026-2
Cada modelo se reentrena con entrenamiento + validación y se evalúa **una sola vez** en 2026-2, un periodo que no se
usó para elegir nada. La logística del Sprint 2 (C = 1, sin ajuste ni NLP) y el control sin NLP también se reentrenan
así, para compararlos en igualdad de condiciones.
- **Error %:** inscritos mal clasificados con umbral de 50%.
- **AUC:** probabilidad de que el modelo ponga a un matriculado por encima de un no matriculado; 0,5 es azar.
- **Brier:** error cuadrático de la probabilidad; más bajo es mejor.

In [ ]:
semilla = cfg["modelado"]["semilla"]
probs = {"Regresión logística (Sprint 2)": mo.construir_pipeline("logistica", cfg["modelado"]["sprint2_logistica"], V2, semilla)
                                             .fit(X2(ev), y(ev)).predict_proba(X2(t))[:, 1],
         "Logística ajustada sin NLP (control)": mo.construir_pipeline("logistica", sin_nlp.iloc[0]["parametros"], V2, semilla)
                                                  .fit(X2(ev), y(ev)).predict_proba(X2(t))[:, 1]}
NOMBRES = {"logistica": "Regresión logística ajustada con NLP", "random_forest": "Random Forest", "xgboost": "XGBoost"}
for tipo in COMPLEJIDAD:
    modelo = mo.construir_pipeline(tipo, mejores[tipo]["parametros"], V, semilla).fit(X(ev), y(ev))
    probs[NOMBRES[tipo]] = modelo.predict_proba(X(t))[:, 1]
res = {n: mo.metricas(y(t), q) for n, q in probs.items()}
comparacion = mo.tabla_metricas(res)
display(comparacion[["n", "error_pct", "auc_roc", "auc_pr", "brier"]])
b, m = res["Regresión logística (Sprint 2)"], res[NOMBRES[elegido]]
dif = mo.diferencia_error(y(t), probs["Regresión logística (Sprint 2)"], probs[NOMBRES[elegido]])
print("\n" + mo.frase_docente(b["error_pct"], m["error_pct"], NOMBRES[elegido], dif, len(t)))
from sklearn.metrics import roc_auc_score
ic = mo.intervalo_bootstrap(y(t), probs[NOMBRES[elegido]], lambda a, q, u: roc_auc_score(a, q))
print(f"AUC del modelo elegido: {m['auc_roc']:.3f} (IC 95% por bootstrap: {ic[0]:.3f} a {ic[1]:.3f})")
c = res["Logística ajustada sin NLP (control)"]
dif_nlp = mo.diferencia_error(y(t), probs["Logística ajustada sin NLP (control)"], probs[NOMBRES["logistica"]])
print(f"Aporte del NLP (misma logística ajustada): error {c['error_pct']:.1f}% → {res[NOMBRES['logistica']]['error_pct']:.1f}% · "
      f"AUC {c['auc_roc']:.3f} → {res[NOMBRES['logistica']]['auc_roc']:.3f} · IC 95% de la diferencia de error: "
      f"{dif_nlp[0]:.1f} a {dif_nlp[1]:.1f} puntos")

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix, roc_curve
fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))
for n, q in probs.items():
    fpr, tpr, _ = roc_curve(y(t), q); ejes[0].plot(fpr, tpr, label=f"{n} ({res[n]['auc_roc']:.3f})")
    fr, me = calibration_curve(y(t), q, n_bins=8, strategy="quantile"); ejes[1].plot(me, fr, "o-", label=n)
ejes[0].plot([0, 1], [0, 1], ":", color=GRIS); ejes[0].set(title="Curva ROC (prueba 2026-2)", xlabel="falsos positivos", ylabel="verdaderos positivos")
ejes[1].plot([0, 1], [0, 1], ":", color=GRIS); ejes[1].set(title="Calibración", xlabel="probabilidad predicha", ylabel="tasa real")
for ax in ejes[:2]:
    ax.legend(fontsize=7)
cm = confusion_matrix(y(t), probs[NOMBRES[elegido]] >= 0.5)
ejes[2].imshow(cm, cmap="Blues")
for (i, j), val in np.ndenumerate(cm):
    ejes[2].text(j, i, val, ha="center", va="center", fontsize=13)
ejes[2].set_xticks([0, 1], ["predice NO", "predice SÍ"]); ejes[2].set_yticks([0, 1], ["no se matriculó", "se matriculó"])
ejes[2].set_title(f"Matriz de confusión · {NOMBRES[elegido]}")
plt.tight_layout(); plt.show()

**Lectura.**
- La frase del docente sale de las cifras y de su intervalo bootstrap pareado: solo afirma una mejora si el intervalo
  de la diferencia de error no incluye el 0.
- La logística con NLP tiene el menor error, el mejor AUC y el mejor Brier, pero su ventaja frente a la logística del
  Sprint 2 está **dentro del margen del azar**: con 761 inscritos de prueba no alcanza para afirmar que es mejor.
- El control sin NLP no mejora a la logística del Sprint 2: ajustar hiperparámetros solo no basta. Lo que mueve el
  error y el AUC es el **texto de la entrevista**, aunque el intervalo de esa diferencia también toca el cero.
- Se reporta tal cual: el modelo se eligió con la validación, antes de mirar la prueba, y no se cambia después.
- Aun así se equivoca en cerca de 1 de cada 3 inscritos: la decisión de matricularse depende de factores que no están
  en los datos (familia, otras ofertas, trabajo).
- Su valor está en **ordenar**: a los de mayor probabilidad se les asegura la matrícula y a los indecisos se les hace
  seguimiento (Sprint 5).
- Las curvas de calibración cerca de la diagonal indican que un 60% predicho significa cerca de 60% real.

## 4. Modelos finales y tabla de predicciones (bloque 3 del tablero)
El modelo elegido se reentrena con **todo el histórico** (2024-1 a 2026-2) para predecir la campaña 2027-1. La tabla
`prediccion` (también en Supabase) tiene, por inscrito, la probabilidad de matrícula, el decil y el segmento de
propensión.

In [ ]:
hist = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
final = mo.construir_pipeline(elegido, mejores[elegido]["parametros"], V, semilla).fit(X(hist), y(hist))
if elegido == "logistica":                      # qué tanto pesa el puntaje NLP en la regresión logística final
    coef = pd.Series(final.named_steps["modelo"].coef_[0], index=final.named_steps["preparacion"].get_feature_names_out())
    print("Coeficientes de la entrevista (variables estandarizadas):",
          coef[["puntaje_vocacional", "puntaje_vocacional_nlp", "entrevista_realizada"]].round(3).to_dict(),
          f"· variables en cero por L1: {int((coef == 0).sum())} de {len(coef)}")
pm = final.predict_proba(X(d))[:, 1].round(4)
s = cfg["modelado"]["segmentos_propension"]
prediccion = pd.DataFrame({"inscripcion_id": d["inscripcion_id"], "periodo": d["periodo"], "prob_matricula": pm,
                           "segmento_propension": np.select([pm >= s["ALTA"], pm < s["BAJA"]], ["ALTA", "BAJA"], "MEDIA")})
display(prediccion[prediccion["periodo"] == "2027-1"].sort_values("prob_matricula", ascending=False).head(10))
print(prediccion.groupby("periodo")["segmento_propension"].value_counts().unstack().fillna(0).astype(int))

## Conclusiones del Sprint 3
1. El modelo de Machine Learning **ya predice** la matrícula de cada inscrito y ordena mejor que el azar.
2. La mejora frente al Sprint 2 viene de **mejores datos**, no de un modelo más complejo: el puntaje vocacional NLP
   (las palabras del aspirante) baja el error y sube el AUC; ajustar hiperparámetros sin él casi no cambia nada.
3. La regresión logística queda por delante de Random Forest y XGBoost: más simple, más exacta y más fácil de explicar.
   Se elige con evidencia, no por preferencia.
4. Los modelos finales quedan guardados (`sprints/sprint_3_modelado/modelos/`). Los sprints 4 a 6 se actualizarán
   con este modelo cuando se retomen.

**FIN DEL SPRINT 3**

---
**COMIENZA EL SPRINT 4** · entrega del jueves 12 de noviembre de 2026 · archivos del sprint en `sprints/sprint_4_explicabilidad/`

# Sprint 4 — ¿Por qué el modelo toma sus decisiones? (Machine Learning explicable)
**Pedido del docente (Semana 4):** un análisis que responda *"¿Por qué el modelo toma sus decisiones?"* con SHAP o
LIME. Por ejemplo, mostrar que el modelo le da más peso al ICFES que a la edad, con una explicación en español sencillo
de por qué eso tiene sentido para la Universidad.

| Sección | Contenido |
|---|---|
| 1 | Qué es SHAP y cómo se agrupan las variables en conceptos de negocio |
| 2 | Importancia global (modelo elegido vs. XGBoost) |
| 3 | Cómo cambia cada factor: descuento, distancia, Saber 11 y canal |
| 4 | Tres inscritos de la campaña 2027-1: SHAP vs. LIME |
| 5 | Chequeo independiente: importancia por permutación |
| 6 | Chequeos de coherencia y equidad por género |
| 7 | ¿El modelo encontró las causas reales? (comparación con el generador) |
| 8 | Explicaciones para el tablero (bloque 4) |

In [ ]:
# Hiperparámetros elegidos en el Sprint 3 (sprints/sprint_3_modelado/modelos/*.json)
PARAMETROS_FINALES = {'matricula': {'parametros': {'C': 1.0, 'l1_ratio': 1.0, 'solver': 'saga'}, 'tipo': 'logistica'},
 'xgboost': {'parametros': {'learning_rate': 0.03,
                            'max_depth': 2,
                            'min_child_weight': 5,
                            'n_estimators': 200}}}

art = flujo.modelos_finales(f, PARAMETROS_FINALES)     # mismo resultado que los .joblib del repositorio
print("Modelos reentrenados:", ", ".join(k for k in art if k.startswith(("modelo", "logistica"))))

## 1. Qué es SHAP
SHAP reparte la predicción de cada inscrito entre sus variables, como un **reparto justo** entre jugadores de un equipo
(valores de Shapley, teoría de juegos). Para cada inscrito:

`log-odds predicho = valor base (inscrito promedio) + suma de las contribuciones de sus variables`

- Una contribución **positiva** sube la probabilidad de matrícula y una **negativa** la baja.
- Como las contribuciones **se suman**, las variables que miden la misma idea se agrupan en conceptos. Por ejemplo,
  la beca, el descuento y el valor neto forman "Beca y precio".
- En la regresión logística, SHAP es exacto: contribución = coeficiente × (valor − promedio).

In [ ]:
import explicabilidad as ex
V = mo.VARIABLES_MATRICULA_NLP          # las del modelo del Sprint 3 (con el puntaje vocacional NLP)
hist = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
X_hist = mo.preparar_X(hist, ref, V)
lm, xgb, fondo = art["modelo_matricula"], art["modelo_matricula_xgboost"], art["fondo_matricula"]
c_lm, base_lm = ex.contribuciones(lm, fondo, X_hist, V)
c_xg, _ = ex.contribuciones(xgb, fondo, X_hist, V)
g_lm, g_xg = ex.agrupar(c_lm, ex.GRUPOS_MATRICULA), ex.agrupar(c_xg, ex.GRUPOS_MATRICULA)
print(f"Valor base (probabilidad de un inscrito promedio): {1 / (1 + np.exp(-base_lm)):.1%}")
# Comprobación de la aditividad: base + suma de contribuciones = predicción del modelo
pred = lm.predict_proba(X_hist)[:, 1]
reconstruida = 1 / (1 + np.exp(-(base_lm + c_lm.sum(axis=1))))
print("Diferencia máxima entre la predicción y base + suma de SHAP:", float(np.abs(pred - reconstruida).max()).__round__(6))
display(pd.DataFrame([(g, ", ".join(v)) for g, v in ex.GRUPOS_MATRICULA.items()], columns=["concepto", "variables"]))

## 2. Importancia global
Importancia = el promedio del **valor absoluto** de la contribución, es decir, cuánto mueve un concepto la predicción
en cualquier sentido. Se compara con XGBoost: si dos modelos distintos señalan lo mismo, la conclusión es más sólida.

In [ ]:
imp = pd.DataFrame({"regresion_logistica": ex.importancia(g_lm), "xgboost": ex.importancia(g_xg)}).sort_values("regresion_logistica", ascending=False)
imp["% del total"] = 100 * imp["regresion_logistica"] / imp["regresion_logistica"].sum()
display(imp.round(3))
fig, ejes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, col, color in [(ejes[0], "regresion_logistica", AZUL), (ejes[1], "xgboost", NARANJA)]:
    s = imp[col].sort_values(); ax.barh(s.index, s.values, color=color); ax.set_title(f"Importancia SHAP · {col}")
plt.tight_layout(); plt.show()
veces = imp.loc["Puntaje Saber 11", "regresion_logistica"] / imp.loc["Edad y años desde el Saber 11", "regresion_logistica"]
print(f"El modelo le da {veces:.1f} veces más peso al puntaje Saber 11 que a la edad.")

**En español sencillo.** Lo que más decide si un inscrito se matricula es **cuánto le cuesta** (beca y descuento),
seguido de cerca por la **entrevista vocacional** (sobre todo lo que dicen sus propias palabras, el puntaje NLP del
Sprint 2). Le siguen el **canal** por el que llegó, su **puntaje Saber 11**, la **distancia** desde su casa al campus y
el **tipo de colegio**. La edad y el género pesan poco. Tiene sentido para la Universidad: quien muestra una vocación clara se
compromete más, y el precio y la cercanía son barreras reales.

La entrevista se lee como **un solo concepto** (si la presentó, el puntaje del entrevistador y el puntaje NLP). Por
separado se compensan: con las mismas palabras, un puntaje alto del entrevistador deja menos intención por explicar, y
el modelo le da un peso negativo. Sumados, dicen cuánto mueve la entrevista.

## 3. Cómo cambia cada factor

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(19, 4))
ejes[0].scatter(X_hist["pct_descuento_inicial"], c_lm[["pct_descuento_inicial", "valor_neto_primer_semestre", "beca"]].sum(axis=1), s=6, alpha=0.4, color=AZUL)
ejes[0].set(title="Beca y descuento", xlabel="descuento 1er semestre", ylabel="contribución SHAP")
ejes[1].scatter(np.exp(X_hist["log_distancia"]), c_lm["log_distancia"], s=6, alpha=0.4, color=AZUL); ejes[1].set(title="Distancia (km)")
ejes[2].scatter(X_hist["punt_global"], c_lm[["punt_global", "icfes_al_cuadrado"]].sum(axis=1), s=6, alpha=0.4, color=AZUL)
ejes[2].axvline(310, color=NARANJA, ls="--"); ejes[2].set(title="Saber 11: U invertida", xlabel="puntaje global")
canal = (c_lm["canal"] + c_lm["captado_en_evento"]).groupby(X_hist["canal"]).mean().sort_values()
ejes[3].barh(canal.index, canal.values, color=[NARANJA if x < 0 else AZUL for x in canal.values]); ejes[3].set(title="Canal (contribución media)")
plt.tight_layout(); plt.show()

## 4. Tres inscritos de la campaña 2027-1: SHAP vs. LIME
LIME explica **un** inscrito ajustando un modelo lineal sencillo con miles de "vecinos" simulados parecidos a él. Si
SHAP y LIME señalan los mismos factores, la explicación es más confiable.

In [ ]:
camp = flujo.conjunto(d, "CAMPANA")
X_camp = mo.preparar_X(camp, ref, V)
prob_camp = pd.Series(lm.predict_proba(X_camp)[:, 1], index=camp.index)
casos = {n: (prob_camp - prob_camp.quantile(q)).abs().idxmin() for n, q in [("ALTA", 0.9), ("MEDIA", 0.5), ("BAJA", 0.1)]}
c_camp, _ = ex.contribuciones(lm, fondo, X_camp.loc[list(casos.values())], V)
g_camp = ex.agrupar(c_camp, ex.GRUPOS_MATRICULA)
for nombre, idx in casos.items():
    print(f"\n=== Propensión {nombre} · inscrito {int(camp.loc[idx, 'inscripcion_id'])} · probabilidad {prob_camp[idx]:.0%}")
    top = g_camp.loc[idx].reindex(g_camp.loc[idx].abs().sort_values(ascending=False).index)[:4]
    print("SHAP (conceptos):", "; ".join(f"{k} {v:+.2f}" for k, v in top.items()))
    lime = ex.explicar_lime(lm, X_hist.sample(800, random_state=2), X_camp.loc[[idx]], V, n_variables=5)
    print("LIME (reglas locales):", "; ".join(f"{k} {w:+.3f}" for k, w in lime))

## 5. Chequeo independiente: importancia por permutación
Se **desordena** un concepto en la prueba 2026-2, dejando lo demás igual, y se mide cuánto cae el AUC del modelo. No
usa SHAP: es otra forma de preguntar de qué depende el modelo.

In [ ]:
ev, t = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION"), flujo.conjunto(d, "PRUEBA")
pm = PARAMETROS_FINALES["matricula"]     # el modelo elegido, reentrenado sin la prueba para medir en ella
lm_ev = mo.construir_pipeline(pm["tipo"], pm["parametros"], V, cfg["modelado"]["semilla"]).fit(
    mo.preparar_X(ev, ref, V), ev["matriculado"].astype(int))
perm = ex.importancia_permutacion(lm_ev, mo.preparar_X(t, ref, V), t["matriculado"].astype(int), ex.GRUPOS_MATRICULA)
display(perm.head(8).round(4))

## 6. Chequeos de coherencia y equidad
¿El modelo se comporta como se espera? Cada afirmación se comprueba con los datos. La equidad se revisa de dos formas: el género debe pesar poco, y el
modelo no debe crear una diferencia entre hombres y mujeres que no exista en los datos reales.

In [ ]:
coh = ex.chequeos_coherencia(X_hist, c_lm, g_lm, lm.predict_proba(X_hist)[:, 1], hist["matriculado"])
display(coh)
print(f"Cumple {int(coh['cumple'].sum())} de {len(coh)} chequeos")

## 7. ¿El modelo encontró las causas reales?
Como los datos son sintéticos, se conocen los efectos **verdaderos** con los que se generó la matrícula
(configuración del generador, `generador.matricula`). Es una prueba que con datos reales no se puede hacer: ¿la
explicación recupera la dirección de cada causa real?

In [ ]:
gen = ex.comparar_con_generador(X_hist, c_lm, cfg)
display(gen)
print(f"Coinciden {int(gen['coincide'].sum())} de {len(gen)} efectos.")

**Lectura.** Los efectos grandes (beca, distancia, U invertida del Saber 11, entrevista, FOES, canal) se recuperan bien.
Falla el orden del estrato (pocos casos en estratos 5 y 6). El promedio del colegio ya no se revisa: dejó de ser una
variable del modelo. **Lección:** una explicación muestra lo que el modelo aprendió, no garantiza la causa.

## 8. Explicaciones para el tablero
Para cada inscrito se generan los **3 factores** que más mueven su probabilidad de matrícula, en español. Esta es la
tabla `explicacion_prediccion` (bloque 4 del tablero).

In [ ]:
X_todos = mo.preparar_X(d, ref, V)
c_todos, _ = ex.contribuciones(lm, fondo, X_todos, V)
explicacion = ex.top_factores(ex.agrupar(c_todos, ex.GRUPOS_MATRICULA), X_todos, d["inscripcion_id"])
print(f"\nTabla del bloque 4: {len(explicacion):,} filas (3 por inscrito). Ejemplo:")
display(explicacion.head(6))

## Conclusiones del Sprint 4
1. **Por qué decide así:** la beca y el precio explican más de una cuarta parte de las decisiones. Les siguen la
   entrevista, el Saber 11 (en U invertida), la distancia y el canal. La edad y el género pesan poco.
2. **Coherencia:** el modelo pasa los chequeos de coherencia y no crea brechas por género.
3. **Rigor:** SHAP recupera la mayoría de las causas reales del generador, y la importancia por permutación lo
   confirma.
4. **Para el tablero:** cada inscrito tiene sus 3 razones principales en español (`explicacion_prediccion`).

**FIN DEL SPRINT 4**

---
**COMIENZA EL SPRINT 5** · entrega del jueves 19 de noviembre de 2026 · archivos del sprint en `sprints/sprint_5_validacion_impacto/`

# Sprint 5 — Validación con datos no vistos e impacto en matrículas
**Pedido del docente (Semana 5):** *tomar un pequeño grupo de datos que el modelo no haya visto antes y evaluar si
acierta o falla*, con *un cálculo estimado de cómo esto ayuda a la Universidad a no perder dinero o estudiantes*.
Además, simular escenarios que muestren cómo el modelo optimiza la captación y la asignación de recursos.

| Sección | Contenido |
|---|---|
| 1 | Se abre el resultado sellado de la campaña 2027-1 |
| 2 | ¿Acierta o falla? |
| 3 | Escenario 1: ¿a quién llamar primero? |
| 4 | Escenario 2: Expo Estudiantes |
| 5 | Sensibilidad de los supuestos y acciones por segmento |

## 1. Se abre el "sobre sellado"
Desde la Semana 1, el resultado de matrícula de los inscritos de la campaña 2027-1 quedó guardado aparte y **no se usó**
para entrenar, ajustar ni elegir modelos. Ahora se lee de Supabase (tabla `resultado_campana_2027_1`, solo lectura);
sin clave de Supabase, de su copia en el repositorio (en Colab, la de GitHub).

> Aclaración honesta: es un resultado **sintético**, generado con las mismas reglas que el resto de la base. Con datos
> reales de Admisiones, este mismo cuaderno se vuelve a ejecutar.

In [ ]:
import datos
import impacto as im
sellado, origen = datos.cargar("resultado_campana_2027_1", "auto")   # Supabase, o su copia (repositorio o GitHub)
camp = flujo.conjunto(d, "CAMPANA").merge(sellado, on="inscripcion_id", how="left", validate="one_to_one")
print(f"Resultado 2027-1 leído de: {origen} · {len(camp)} inscritos · se matricularon {int(camp['matriculado_real'].sum())}")

## 2. ¿Acierta o falla?

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
V, V2 = mo.VARIABLES_MATRICULA_NLP, mo.VARIABLES_MATRICULA   # elegido y XGBoost con NLP · Sprint 2 sin NLP
Xc, Xc2, yc = mo.preparar_X(camp, ref, V), mo.preparar_X(camp, ref, V2), camp["matriculado_real"].astype(int)
hist = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
probs = {"Modelo elegido (regresión logística con NLP)": art["modelo_matricula"].predict_proba(Xc)[:, 1],
         "XGBoost": art["modelo_matricula_xgboost"].predict_proba(Xc)[:, 1],
         "Regresión logística (Sprint 2)": art["logistica_sprint2"].predict_proba(Xc2)[:, 1]}
res = {n: mo.metricas(yc, q) for n, q in probs.items()}
display(mo.tabla_metricas(res)[["n", "error_pct", "auc_roc", "brier", "precision", "recall"]])
p_final = probs["Modelo elegido (regresión logística con NLP)"]
ic = mo.intervalo_bootstrap(yc, p_final, lambda a, q, u: 100 * ((q >= u) != a).mean())
print(f"Error del modelo elegido: {res['Modelo elegido (regresión logística con NLP)']['error_pct']:.1f}% (IC 95%: {ic[0]:.1f}% a {ic[1]:.1f}%)")
dif = mo.diferencia_error(yc, probs["Regresión logística (Sprint 2)"], p_final)
print(f"Diferencia con la logística del Sprint 2: IC 95% de {dif[0]:.1f} a {dif[1]:.1f} puntos"
      + (" (incluye el 0: rinden igual)." if dif[0] <= 0 <= dif[1] else "."))
camp["prob"] = p_final
q = camp.assign(grupo=pd.qcut(camp["prob"].rank(method="first"), 5, labels=["Q1 menor", "Q2", "Q3", "Q4", "Q5 mayor"])) \
        .groupby("grupo", observed=True).agg(inscritos=("prob", "size"), prob_media=("prob", "mean"), tasa_real=("matriculado_real", "mean"))
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(q))
ejes[0].bar(x - 0.2, q["prob_media"], 0.4, color=AZUL, label="probabilidad predicha")
ejes[0].bar(x + 0.2, q["tasa_real"], 0.4, color=NARANJA, label="tasa real")
ejes[0].set_xticks(x, q.index); ejes[0].legend(fontsize=8); ejes[0].set_title("Predicho vs. real por quintil")
cm = confusion_matrix(yc, p_final >= 0.5); ejes[1].imshow(cm, cmap="Blues")
for (i, j), val in np.ndenumerate(cm):
    ejes[1].text(j, i, val, ha="center", va="center", fontsize=13)
ejes[1].set_xticks([0, 1], ["predice NO", "predice SÍ"]); ejes[1].set_yticks([0, 1], ["no se matriculó", "se matriculó"])
plt.tight_layout(); plt.show()
display(q.round(3))

**Lectura.** El modelo falla en cerca de 1 de cada 3 casos individuales, pero **ordena**: el quintil de mayor
probabilidad se matricula más del doble que el de menor. Para la Universidad, lo valioso es saber **a quién
dedicarle el esfuerzo**, no adivinar a cada persona.

## 3. Escenario 1: ¿a quién llamar primero?
Los asesores solo alcanzan a hacer seguimiento intensivo al 30% de los inscritos (supuesto). La llamada ayuda poco
a quien ya decidió y más al indeciso (supuesto):

`efecto(p) = 0,10 × 4 × p × (1 − p)`, es decir, +10 puntos si p = 0,5 y casi 0 si p está cerca de 0 o de 1.

Valor de una matrícula adicional = precio neto del 1er semestre con su beca (tarifas oficiales). No se calculan costos
porque no hay datos de cuánto cuesta el seguimiento.

In [ ]:
s = cfg["impacto"]["seguimiento"]
e1 = im.escenario_seguimiento(camp["prob"], camp["valor_neto_primer_semestre"], s["capacidad_pct"], s["efecto_maximo"])
display(e1.round(1))
print(f"Llamar a los indecisos (modelo) deja {e1.iloc[2]['matriculas_adicionales'] - e1.iloc[0]['matriculas_adicionales']:.1f} matrículas y "
      f"${e1.iloc[2]['ingreso_primer_semestre_cop'] - e1.iloc[0]['ingreso_primer_semestre_cop']:,.0f} ".replace(",", ".")
      + "más que llamar al azar en una campaña; llamar primero a los de mayor probabilidad es lo peor.")

## 4. Escenario 2: Expo Estudiantes
Se aplica el mismo seguimiento priorizado a los inscritos que llegaron por la Expo en cada campaña histórica y se mide
cuántas **matrículas más** se esperan. No se calcula el costo por matriculado porque no hay datos de la inversión en el
evento.

In [ ]:
expo = hist[hist["canal"] == "EXPOESTUDIANTES"].copy()
expo["prob"] = art["modelo_matricula"].predict_proba(mo.preparar_X(expo, ref, V))[:, 1]
filas = []
for per, g in expo.groupby("periodo"):
    reales = int(g["matriculado"].astype(int).sum())
    t = im.escenario_seguimiento(g["prob"], g["valor_neto_primer_semestre"], s["capacidad_pct"], s["efecto_maximo"])
    filas.append({"campaña": per, "inscritos Expo": len(g), "matriculados": reales,
                  "matrículas adicionales": t.iloc[2]["matriculas_adicionales"]})
display(pd.DataFrame(filas).round(1))
seg = expo.assign(m=expo["matriculado"].astype(float)).groupby("segmento_mercado").agg(inscritos=("m", "size"), tasa=("m", "mean"))
print("Segmentos que mejor convierten cuando llegan por la Expo:")
display(seg.sort_values("tasa", ascending=False).round(3))

## 5. Sensibilidad y acciones por segmento
Cada supuesto se mueve −50% y +50%, dejando los demás fijos, para ver cuánto cambia el resultado (gráfico de tornado).
El supuesto que más mueve la cifra es el que primero hay que validar con Admisiones.

In [ ]:
def matriculas_seguimiento(capacidad_pct, efecto_maximo):
    t = im.escenario_seguimiento(camp["prob"], camp["valor_neto_primer_semestre"], capacidad_pct, efecto_maximo)
    return {"matriculas_con_modelo": float(t.iloc[2]["matriculas_adicionales"])}
sens1 = im.sensibilidad(matriculas_seguimiento, dict(s), list(s), "matriculas_con_modelo")
display(sens1.round(2))
acciones = im.acciones_por_segmento(camp, camp["prob"], cfg["modelado"]["segmentos_propension"])
display(acciones.round(3))

## Conclusiones del Sprint 5
1. Con datos que nunca vio, el modelo **ordena** a los inscritos mucho mejor que el azar (AUC cercano a 0,68). Su error
   individual (cerca de 34%) es unos 3 puntos menor que el de la regresión logística del Sprint 2, pero la diferencia
   todavía cabe en el margen del azar.
2. **Captación:** priorizar a los indecisos da más matrículas e ingreso que llamar al azar, y mucho más que llamar a los
   seguros. En la Expo se esperan algunas matrículas más con el mismo seguimiento.
3. Las cifras dependen de dos supuestos (capacidad de seguimiento y efecto de la llamada). El análisis de sensibilidad
   muestra cuál validar primero. No se incluyen costos porque no hay datos de ellos.

**FIN DEL SPRINT 5**